# Prodrugs
### PHS 312 - Drug Discovery, Design, and Development
**Lecture 32 · Wednesday, November 18, 2026 · 50 minutes**

Pre-class reading: Patrick 7e, Section 14.6

**This activity takes 30-35 minutes.** Work in pairs. Run each cell with the
play button on its left, answer the question that follows, and keep going.
You will not write any code.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Spinner101/phs312-cadd-notebooks/blob/main/notebooks/PHS312_L32_Prodrugs.ipynb)

> **New:** Parts 2–4 are **lecture notes**: key ideas, annotated 2D structures (each prodrug drawn above its active drug), worked examples and calculators. Read them before class or use them afterwards for review. The timed pair activity begins at Part 5; Part 6 is a reference table of marketed prodrugs.

---
### Before you start - what this notebook does with your work

* **Nothing you type is sent anywhere.** This notebook makes no network
  calls, writes no files, and does not connect to Google Drive. Your answers
  live in the memory of this browser tab and disappear when you close it.
* **Your instructor cannot see this page.** At the end you will get a short
  report code. You choose to paste it into Canvas - that is the only thing
  that reaches your instructor.
* **Do not type anything personal.** No patient information, no identifiers,
  nothing about a real person. Everything here is simulated.
* Opening this notebook signs you in with your Hampton Google account, which
  is covered by the university's agreement with Google. If you would rather
  not use a Google account, tell your instructor and you will be given an
  offline copy.
---

In [ ]:
#@title Set-up  -  run this cell first { display-mode: "form" }
#@markdown Press the play button once. Nothing is downloaded and nothing
#@markdown leaves this browser tab. Takes a few seconds.
import sys, types, textwrap

_tk = types.ModuleType("toolkit"); _tk.__dict__["__name__"] = "toolkit"
exec(compile("\"\"\"PHS 312 activity toolkit - assessment engine and simulation helpers.\n\nEverything runs in the browser session. Nothing is transmitted anywhere:\nno network calls, no file writes, no analytics. Scores exist only in the\nstudent's own Colab session until they choose to report them in Canvas.\n\"\"\"\nimport textwrap, hashlib, json\nfrom datetime import datetime\n\nW = 86\ndef _wrap(s, indent=\"\"):\n    out = []\n    for para in str(s).split(\"\\n\"):\n        out.extend(textwrap.wrap(para, W, initial_indent=indent,\n                                 subsequent_indent=indent) or [\"\"])\n    return \"\\n\".join(out)\n\ndef _rule(ch=\"-\"):\n    return ch * W\n\n\nclass Item:\n    \"\"\"One multiple-choice item with per-distractor feedback.\"\"\"\n    def __init__(self, qid, stem, options, key, why, bloom=\"\", silo=\"\",\n                 section=\"concept\", topic=\"\"):\n        if key not in options:\n            raise ValueError(f\"{qid}: key {key!r} not among options {list(options)}\")\n        missing = [k for k in options if k != key and k not in why]\n        if missing:\n            raise ValueError(f\"{qid}: no feedback written for distractor(s) {missing}\")\n        self.qid, self.stem, self.options = qid, stem, options\n        self.key, self.why = key, why\n        self.bloom, self.silo, self.section = bloom, silo, section\n        self.topic = topic\n\n\nclass Gradebook:\n    \"\"\"Tracks responses and reports performance. Local to this session.\"\"\"\n\n    SECTIONS = (\"precheck\", \"concept\", \"post\")\n    LABELS = {\"precheck\": \"Prior-knowledge check\",\n              \"concept\": \"Concept checks during the activity\",\n              \"post\": \"End-of-activity assessment\"}\n\n    def __init__(self, lecture, title):\n        self.lecture, self.title = lecture, title\n        self.items, self.responses = {}, {}\n        self.started = datetime.now()\n\n    # ---------- authoring ----------\n    def add(self, item):\n        if item.qid in self.items:\n            raise ValueError(f\"duplicate qid {item.qid}\")\n        self.items[item.qid] = item\n\n    def add_many(self, items):\n        for it in items:\n            self.add(it)\n\n    # ---------- student interaction ----------\n    def ask(self, qid):\n        \"\"\"Print the stem and options for an item.\"\"\"\n        it = self.items[qid]\n        print(_rule(\"=\"))\n        print(_wrap(f\"{qid}.  {it.stem}\"))\n        print()\n        for letter, text in it.options.items():\n            print(_wrap(f\"{letter}) {text}\", indent=\"    \"))\n        print(_rule(\"=\"))\n\n    def answer(self, qid, choice):\n        \"\"\"Record a response and give feedback appropriate to the section.\"\"\"\n        if qid not in self.items:\n            print(f\"[!] Unknown question id {qid!r}. Check the notebook.\")\n            return\n        it = self.items[qid]\n        choice = (choice or \"\").strip().upper()\n        if choice in (\"\", \"-\", \"SELECT\", \"NONE\"):\n            print(\"Choose an option from the dropdown, then press the play button again.\")\n            return\n        if choice not in it.options:\n            print(f\"[!] {choice!r} is not one of {list(it.options)}.\")\n            return\n\n        first = qid not in self.responses\n        if first:\n            self.responses[qid] = choice      # only the first response scores\n        correct = choice == it.key\n\n        if it.section == \"precheck\":\n            # No feedback yet: revealing it now would teach the answer\n            # before the activity and destroy the pre/post comparison.\n            n = len([q for q in self.responses\n                     if self.items[q].section == \"precheck\"])\n            total = len([q for q in self.items\n                         if self.items[q].section == \"precheck\"])\n            print(f\"Recorded. ({n} of {total} answered)\")\n            if n == total:\n                print(\"\\nAll set - run the next cell to begin the activity.\")\n            return\n\n        print(_rule())\n        if correct:\n            print(f\"Correct - {choice}.\")\n            print(_wrap(it.why.get(\"_correct\", \"\"), \"\"))\n        else:\n            print(f\"Not quite. You chose {choice}.\")\n            print(_wrap(it.why[choice]))\n            print()\n            print(_wrap(f\"The defensible answer is {it.key}. \"\n                        + it.why.get(\"_correct\", \"\")))\n        if not first:\n            kept = self.responses[qid]\n            print(f\"\\n(Your first response, {kept}, is the one on record - \"\n                  f\"this retry is for your own understanding.)\")\n        print(_rule())\n\n    # ---------- reporting ----------\n    def _score(self, section):\n        qs = [q for q, it in self.items.items() if it.section == section]\n        got = sum(1 for q in qs if self.responses.get(q) == self.items[q].key)\n        return got, len(qs)\n\n    def summary(self, show_pre_post=True):\n        print(_rule(\"=\"))\n        print(f\"  {self.lecture} - {self.title}\")\n        print(f\"  Performance summary\")\n        print(_rule(\"=\"))\n        any_missing = False\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            if tot == 0:\n                continue\n            answered = len([q for q, it in self.items.items()\n                            if it.section == sec and q in self.responses])\n            if answered < tot:\n                any_missing = True\n            pct = 100 * got / tot if tot else 0\n            bar = \"#\" * int(round(pct / 5)) + \".\" * (20 - int(round(pct / 5)))\n            print(f\"  {self.LABELS[sec]:<38} {got:>2}/{tot:<3} {pct:5.1f}%  [{bar}]\")\n            if answered < tot:\n                print(f\"      ({tot - answered} unanswered)\")\n        print(_rule())\n\n        if show_pre_post:\n            pre_g, pre_t = self._score(\"precheck\")\n            post_g, post_t = self._score(\"post\")\n            if pre_t and post_t:\n                pre = 100 * pre_g / pre_t\n                post = 100 * post_g / post_t\n                print(f\"  Before the activity: {pre:5.1f}%     \"\n                      f\"After the activity: {post:5.1f}%\")\n                if post >= pre:\n                    if pre < 100:\n                        g = (post - pre) / (100 - pre)\n                        print(f\"  Normalised learning gain  g = {g:.2f}\"\n                              f\"   {self._gain_note(g)}\")\n                    else:\n                        print(\"  You came in at ceiling on the prior check.\")\n                else:\n                    print(\"  Your post score is below your pre score - look back at the\")\n                    print(\"  concept checks you missed before you leave.\")\n                print(_rule())\n\n        self._missed()\n        if any_missing:\n            print(\"\\n  Some questions are unanswered. Scroll back and complete them\")\n            print(\"  before copying your report code.\")\n        return self\n\n    @staticmethod\n    def _gain_note(g):\n        if g >= 0.7: return \"(high)\"\n        if g >= 0.3: return \"(medium)\"\n        return \"(low - review before the exam)\"\n\n    def _missed(self):\n        missed = [(q, it) for q, it in self.items.items()\n                  if it.section != \"precheck\"\n                  and q in self.responses and self.responses[q] != it.key]\n        if not missed:\n            print(\"  Nothing missed in the graded sections.\")\n            return\n        print(\"  Worth reviewing before the exam:\")\n        for q, it in missed:\n            tag = f\"[{it.bloom}\" + (f\" | SILO {it.silo}\" if it.silo else \"\") + \"]\"\n            print(_wrap(f\"- {q} {tag}: {it.stem}\", indent=\"    \"))\n\n    def report_code(self, student_name=\"\"):\n        \"\"\"A short code the student pastes into Canvas.\n\n        This is a participation signal, not a secure exam score: it is\n        generated in the browser and a determined student could edit it.\n        Weight it accordingly.\n        \"\"\"\n        tags = {\"precheck\": \"PRE\", \"concept\": \"CON\", \"post\": \"POST\"}\n        parts = []\n        for sec in self.SECTIONS:\n            got, tot = self._score(sec)\n            parts.append(f\"{tags[sec]}{got}/{tot}\" if tot else f\"{tags[sec]}-\")\n        body = f\"{self.lecture}|\" + \",\".join(parts)\n        digest = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper()\n        code = f\"{body}|{digest}\"\n        print(_rule(\"=\"))\n        print(\"  COPY THIS LINE INTO THE CANVAS ASSIGNMENT\")\n        print(_rule(\"=\"))\n        print(f\"  {code}\")\n        print(_rule(\"=\"))\n        print(_wrap(\"Nothing was sent anywhere. This code exists only in your \"\n                    \"browser until you paste it into Canvas yourself.\"))\n        return code\n\n\ndef verify_code(code):\n    \"\"\"Instructor-side check that a pasted code has not been casually edited.\"\"\"\n    try:\n        body, digest = code.strip().rsplit(\"|\", 1)\n        ok = hashlib.sha256((body + \"|phs312\").encode()).hexdigest()[:4].upper() == digest.upper()\n        return ok, body\n    except Exception:\n        return False, \"\"\n\n\n    # ---------- diagnostic review (used by the L28 review session) ----------\ndef _topic_table(gb, guide=None, section=None):\n    \"\"\"Score by topic and say what to revisit. Attached to Gradebook below.\n\n    `section` limits the table to one section, so the mid-session diagnostic\n    does not count questions the student has not reached yet.\n    \"\"\"\n    rows = {}\n    for qid, it in gb.items.items():\n        if not it.topic:\n            continue\n        if section and it.section != section:\n            continue\n        got = gb.responses.get(qid)\n        r = rows.setdefault(it.topic, [0, 0, 0])\n        r[1] += 1\n        if got is not None:\n            r[2] += 1\n            if got == it.key:\n                r[0] += 1\n    if not rows:\n        print(\"No topic-tagged questions have been answered yet.\")\n        return {}\n    print(_rule(\"=\"))\n    print(\"  Where you stand, by lecture\")\n    print(_rule(\"=\"))\n    print(f\"  {'lecture':<9}{'answered':>10}{'correct':>9}{'':>4}\")\n    weak = []\n    for t in sorted(rows):\n        ok, total, ans = rows[t]\n        pct = (100.0 * ok / ans) if ans else 0.0\n        bar = \"#\" * int(round(pct / 10)) + \".\" * (10 - int(round(pct / 10)))\n        flag = \"\"\n        if ans == 0:\n            bar, flag = \"-\" * 10, \"  not attempted\"\n        elif pct < 67:\n            flag = \"  <- revisit\"\n            weak.append(t)\n        print(f\"  {t:<9}{ans:>5} of {total:<4}{ok:>5}   {bar}{flag}\")\n    print()\n    if weak:\n        print(\"  Before the exam, go back to:\")\n        for t in weak:\n            where = (guide or {}).get(t, \"\")\n            print(f\"    {t}  {where}\" if where else f\"    {t}\")\n    else:\n        done = sum(r[2] for r in rows.values())\n        total = sum(r[1] for r in rows.values())\n        if done < total:\n            print(f\"  Nothing below two thirds so far, with {total - done} \"\n                  f\"question(s) still unanswered.\")\n        else:\n            print(\"  No lecture is below two thirds. Review your missed \"\n                  \"questions above,\\n  then re-run the activities you found \"\n                  \"slowest rather than re-reading.\")\n    print(_rule(\"=\"))\n    return {t: dict(correct=v[0], answered=v[2], total=v[1]) for t, v in rows.items()}\n\n\nGradebook.topic_report = _topic_table\n", "toolkit.py", "exec"), _tk.__dict__)
sys.modules["toolkit"] = _tk
Item, Gradebook, verify_code = _tk.Item, _tk.Gradebook, _tk.verify_code

_sb = types.ModuleType("swissbio"); _sb.__dict__["__name__"] = "swissbio"
exec(compile("# -*- coding: utf-8 -*-\n\"\"\"Curated bioisosteric replacement data for PHS 312 Lecture 31.\n\nSOURCE\n    SwissBioisostere, https://www.swissbioisostere.ch\n    Molecular Modelling Group, SIB Swiss Institute of Bioinformatics and\n    the University of Lausanne. Underlying data derived from ChEMBL 36.\n\nLICENCE\n    CC-BY 4.0  https://creativecommons.org/licenses/by/4.0/\n    Redistribution and adaptation permitted with attribution.\n\nCHANGES MADE (required by CC-BY)\n    Retrieved 20 October 2026 through the SwissBioisostere web interface.\n    Two single-attachment-point (\"side chain\") fragment queries were run:\n    carboxylic acid [*:1]C(=O)O  and primary amide [*:1]C(=O)N.\n    The full result tables (7,714 and 6,955 replacements) were reduced to the\n    highest-frequency entries shown below. Column names were shortened and\n    values were not otherwise altered. No structures were recomputed.\n\nCITE\n    Cuozzo A., Daina A., Perez M.A.S., Michielin O., Zoete V.\n    SwissBioisostere: molecular replacements for drug discovery guided by\n    updated structural, bioactivity and physico-chemical data accessible\n    through a reshaped Web interface. Nucleic Acids Research, 2021.\n    Wirth M., Zoete V., Michielin O., Sauer W.H.B. SwissBioisostere: a\n    database of molecular replacements for ligand design. Nucleic Acids\n    Research 2013, 41(D1), D1137-D1143. doi:10.1093/nar/gks1059\n\nCOLUMNS\n    smiles  replacement fragment, [*:1] marks the attachment point\n    freq    number of matched molecular pairs in which this replacement occurs\n    rank    rank by frequency within that query\n    better / similar / worse\n            how the replacement changed measured activity across those pairs\n    dlogp, dtpsa, dmw\n            mean change in calculated logP, topological PSA and molecular\n            weight on making the replacement\n\"\"\"\n\nFIELDS = (\"smiles\", \"freq\", \"rank\", \"better\", \"similar\", \"worse\",\n          \"dlogp\", \"dtpsa\", \"dmw\")\n\n# common names for the fragments students meet in Patrick 7e 13.3.7\nNAMES = {\n    \"COC(=O)[*:1]\": \"methyl ester\",\n    \"NC(=O)[*:1]\": \"primary amide\",\n    \"CCOC(=O)[*:1]\": \"ethyl ester\",\n    \"O[*:1]\": \"hydroxyl\",\n    \"C[*:1]\": \"methyl\",\n    \"OC[*:1]\": \"hydroxymethyl\",\n    \"O=C(O)C[*:1]\": \"acetic acid (homologated)\",\n    \"N#C[*:1]\": \"nitrile\",\n    \"CNC(=O)[*:1]\": \"N-methyl amide\",\n    \"CO[*:1]\": \"methoxy\",\n    \"N[*:1]\": \"amine\",\n    \"CN(C)C(=O)[*:1]\": \"N,N-dimethyl amide\",\n    \"O=C(NO)[*:1]\": \"hydroxamic acid\",\n    \"c1ccc([*:1])cc1\": \"phenyl\",\n    \"F[*:1]\": \"fluorine\",\n    \"Cl[*:1]\": \"chlorine\",\n    \"n1n[nH]c([*:1])n1\": \"tetrazol-5-yl\",\n    \"FC(F)(F)[*:1]\": \"trifluoromethyl\",\n    \"n1nc([*:1])n[nH]1\": \"tetrazol-5-yl (tautomer)\",\n    \"O=[N+]([O-])[*:1]\": \"nitro\",\n    \"NS(=O)(=O)[*:1]\": \"sulfonamide\",\n    \"CS(=O)(=O)NC(=O)[*:1]\": \"acylsulfonamide\",\n    \"O=C(O)CC[*:1]\": \"propanoic acid (homologated)\",\n    \"O=C(NCCO)[*:1]\": \"N-(2-hydroxyethyl) amide\",\n    \"CC(=O)N[*:1]\": \"acetamido\",\n    \"Br[*:1]\": \"bromine\",\n    \"CS(=O)(=O)[*:1]\": \"methylsulfonyl\",\n    \"O=C(N1CCOCC1)[*:1]\": \"morpholine amide\",\n    \"CC(=O)[*:1]\": \"acetyl\",\n    \"CN(C)[*:1]\": \"dimethylamino\",\n    \"O=C(O)[*:1]\": \"carboxylic acid\",\n    \"CCNC(=O)[*:1]\": \"N-ethyl amide\",\n    \"O=C(NC1CC1)[*:1]\": \"N-cyclopropyl amide\",\n    \"CS(=O)(=O)N[*:1]\": \"methanesulfonamido\",\n}\n\n# ---- carboxylic acid: what replaces [*:1]C(=O)O  (7,714 replacements total)\nACID_TOTAL = 7714\nACID = [\n (\"COC(=O)[*:1]\", 4530, 1, 1238, 2353, 939, 0.09, -11.00, 14.03),\n (\"NC(=O)[*:1]\", 2781, 2, 1000, 1334, 447, -0.60, 5.79, -0.99),\n (\"CCOC(=O)[*:1]\", 2449, 3, 703, 1024, 722, 0.48, -11.00, 28.06),\n (\"O[*:1]\", 1626, 4, 604, 793, 229, 0.01, -17.07, -28.01),\n (\"C[*:1]\", 1478, 5, 620, 616, 242, 0.61, -37.30, -29.98),\n (\"OC[*:1]\", 1144, 6, 395, 546, 203, -0.20, -17.07, -13.98),\n (\"O=C(O)C[*:1]\", 956, 7, 253, 530, 173, -0.08, 0.00, 14.02),\n (\"N#C[*:1]\", 877, 8, 346, 353, 178, 0.17, -13.51, -19.00),\n (\"CNC(=O)[*:1]\", 818, 9, 298, 412, 108, -0.34, -8.20, 13.04),\n (\"CO[*:1]\", 788, 10, 331, 334, 123, 0.31, -28.07, -13.98),\n (\"N[*:1]\", 721, 11, 201, 368, 152, -0.51, -11.28, -29.00),\n (\"CN(C)C(=O)[*:1]\", 692, 12, 230, 331, 131, 0.00, -16.99, 27.07),\n (\"O=C(NO)[*:1]\", 654, 13, 440, 157, 57, -0.58, 12.03, 15.02),\n (\"c1ccc([*:1])cc1\", 635, 14, 271, 262, 102, 1.96, -37.30, 32.09),\n (\"F[*:1]\", 625, 15, 265, 259, 101, 0.44, -37.30, -26.02),\n (\"Cl[*:1]\", 604, 16, 276, 240, 88, 0.95, -37.30, -9.57),\n (\"n1n[nH]c([*:1])n1\", 545, 17, 105, 325, 115, -0.52, 17.16, 24.03),\n (\"FC(F)(F)[*:1]\", 503, 18, 214, 206, 83, 1.32, -37.30, 23.98),\n (\"n1nc([*:1])n[nH]1\", 484, 19, 135, 256, 93, -0.51, 17.16, 24.03),\n (\"O=[N+]([O-])[*:1]\", 384, 20, 166, 167, 51, 0.21, 5.84, 0.99),\n (\"NS(=O)(=O)[*:1]\", 382, 21, 157, 168, 57, -1.05, 22.86, 35.07),\n (\"CS(=O)(=O)NC(=O)[*:1]\", 369, 22, 36, 226, 107, -1.01, 25.94, 77.11),\n (\"O=C(O)CC[*:1]\", 348, 23, 112, 191, 45, 0.32, 0.00, 28.06),\n (\"O=C(NCCO)[*:1]\", 341, 24, 148, 142, 51, -0.98, 12.03, 43.07),\n (\"CC(=O)N[*:1]\", 331, 25, 111, 160, 60, 0.26, -8.20, 13.04),\n (\"Br[*:1]\", 330, 26, 153, 125, 52, 1.06, -37.30, 34.89),\n (\"CS(=O)(=O)[*:1]\", 327, 27, 132, 141, 54, -0.43, -3.16, 34.08),\n (\"O=C(N1CCOCC1)[*:1]\", 315, 28, 93, 151, 71, -0.23, -7.76, 69.10),\n (\"CC(=O)[*:1]\", 296, 29, 120, 118, 58, 0.50, -20.23, -1.97),\n (\"CN(C)[*:1]\", 283, 30, 93, 116, 74, 0.08, -34.06, -0.94),\n]\n\n# ---- primary amide: what replaces [*:1]C(=O)N  (6,955 replacements total)\nAMIDE_TOTAL = 6955\nAMIDE = [\n (\"O=C(O)[*:1]\", 2781, 1, 447, 1334, 1000, 0.60, -5.79, 0.99),\n (\"N#C[*:1]\", 2384, 2, 586, 1189, 609, 1.04, -19.30, -18.01),\n (\"CNC(=O)[*:1]\", 2202, 3, 192, 1591, 419, 0.26, -13.99, 14.03),\n (\"O[*:1]\", 1622, 4, 398, 1003, 221, 0.11, -22.86, -27.02),\n (\"COC(=O)[*:1]\", 1269, 5, 259, 699, 311, 0.68, -16.79, 15.02),\n (\"C[*:1]\", 1238, 6, 361, 668, 209, 1.21, -43.09, -28.99),\n (\"CN(C)C(=O)[*:1]\", 1112, 7, 128, 590, 394, 0.61, -22.78, 28.06),\n (\"OC[*:1]\", 1060, 8, 233, 651, 176, 0.51, -22.86, -13.00),\n (\"CO[*:1]\", 1003, 9, 235, 554, 214, 0.91, -33.86, -12.99),\n (\"F[*:1]\", 720, 10, 195, 360, 165, 1.04, -43.09, -25.04),\n (\"Cl[*:1]\", 669, 11, 262, 281, 126, 1.55, -43.09, -8.58),\n (\"N[*:1]\", 646, 12, 151, 332, 163, 0.08, -17.07, -28.01),\n (\"FC(F)(F)[*:1]\", 635, 13, 194, 272, 169, 1.92, -43.09, 24.97),\n (\"CCOC(=O)[*:1]\", 612, 14, 146, 344, 122, 1.08, -16.79, 29.03),\n (\"CS(=O)(=O)[*:1]\", 599, 15, 90, 403, 106, 0.17, -8.95, 35.06),\n (\"CC(=O)N[*:1]\", 546, 16, 80, 349, 117, 0.26, -13.99, 14.03),\n (\"O=C(NCCO)[*:1]\", 532, 17, 42, 385, 105, -0.38, 6.24, 44.05),\n (\"c1ccc([*:1])cc1\", 476, 18, 128, 246, 102, 2.37, -43.09, 33.07),\n (\"O=C(N1CCOCC1)[*:1]\", 443, 19, 45, 250, 148, 0.37, -13.55, 70.09),\n (\"CC(=O)[*:1]\", 441, 20, 112, 235, 94, 1.10, -26.02, -0.98),\n (\"CCNC(=O)[*:1]\", 438, 21, 65, 269, 104, 0.65, -13.99, 28.06),\n (\"O=C(NC1CC1)[*:1]\", 431, 22, 94, 246, 91, 0.80, -13.99, 40.06),\n (\"NS(=O)(=O)[*:1]\", 382, 23, 72, 232, 78, -0.45, 17.07, 36.06),\n (\"CS(=O)(=O)N[*:1]\", 359, 24, 58, 225, 76, 0.27, 3.08, 50.08),\n (\"Br[*:1]\", 339, 25, 137, 146, 56, 1.66, -43.09, 35.88),\n]\n\nQUERIES = {\n    \"carboxylic acid\": dict(query=\"[*:1]C(=O)O\", total=ACID_TOTAL, rows=ACID),\n    \"primary amide\":   dict(query=\"[*:1]C(=O)N\", total=AMIDE_TOTAL, rows=AMIDE),\n}\n\n\ndef table(which=\"carboxylic acid\"):\n    q = QUERIES[which]\n    return [dict(zip(FIELDS, r), name=NAMES.get(r[0], r[0])) for r in q[\"rows\"]]\n\n\ndef success_rate(row):\n    \"\"\"Fraction of matched pairs where activity did not get worse.\"\"\"\n    n = row[\"better\"] + row[\"similar\"] + row[\"worse\"]\n    return (row[\"better\"] + row[\"similar\"]) / n if n else 0.0\n", "swissbio.py", "exec"), _sb.__dict__)
sys.modules["swissbio"] = _sb
swissbio = _sb


_sm = types.ModuleType("sims"); _sm.__dict__["__name__"] = "sims"
exec(compile("\"\"\"PHS 312 simulations. NumPy + matplotlib only - no install, no network.\n\nEvery function returns the numbers it plotted so the notebook can quiz on them.\n\"\"\"\nimport numpy as np\nimport matplotlib\nimport matplotlib.pyplot as plt\n\nPRIMARY, SECOND, MUTED = \"#0B6E8F\", \"#C2410C\", \"#6B7280\"\n\n\ndef _tw(text, width=76, indent=\"\"):\n    import textwrap\n    return \"\\n\".join(textwrap.fill(p, width, initial_indent=indent,\n                                   subsequent_indent=indent)\n                     for p in text.split(\"\\n\"))\n\n\ndef _fig(w=9.5, h=4.4):\n    f, ax = plt.subplots(figsize=(w, h))\n    ax.grid(alpha=.25); ax.set_axisbelow(True)\n    for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    return f, ax\n\n\n# ---------------------------------------------------------------- L22\ndef force_field_terms(bond_k=300., bond_r0=1.52, ang_k=60., ang_t0=109.5,\n                      tor_V=14., vdw_eps=0.3, vdw_sigma=3.6, q1q2=-0.3):\n    \"\"\"Each force-field term plotted over its own coordinate.\"\"\"\n    f, axes = plt.subplots(1, 4, figsize=(15, 3.4))\n    r = np.linspace(1.1, 2.1, 200); Es = bond_k * (r - bond_r0) ** 2\n    t = np.linspace(80, 140, 200);  Eb = ang_k * np.radians(t - ang_t0) ** 2\n    d = np.linspace(-180, 180, 361); Et = tor_V / 2 * (1 + np.cos(np.radians(3 * d)))\n    x = np.linspace(3.0, 8.0, 300)\n    Ev = 4 * vdw_eps * ((vdw_sigma / x) ** 12 - (vdw_sigma / x) ** 6)\n    Ec = 332. * q1q2 / x\n    for ax, (xx, yy, xl, ti) in zip(axes, [\n            (r, Es, \"bond length (A)\", \"stretch\"),\n            (t, Eb, \"bond angle (deg)\", \"bend\"),\n            (d, Et, \"torsion (deg)\", \"torsion\"),\n            (x, Ev + Ec, \"separation (A)\", \"vdW + electrostatic\")]):\n        ax.plot(xx, yy, color=PRIMARY, lw=2)\n        ax.set_title(ti, fontsize=10); ax.set_xlabel(xl, fontsize=8)\n        ax.grid(alpha=.25); ax.axhline(0, color=MUTED, lw=.8)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    axes[0].set_ylabel(\"energy (kcal/mol)\", fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(stretch_min=float(r[Es.argmin()]), torsion_barrier=float(Et.max()),\n                vdw_min_sep=float(x[(Ev + Ec).argmin()]),\n                vdw_min_E=float((Ev + Ec).min()))\n\n\n# ---------------------------------------------------------------- L23\ndef torsion_profile(V3=14.0, gauche_strain=3.8, step=10, start_angle=100.):\n    \"\"\"Butane-like relaxed torsion profile, plus where minimisation stops.\"\"\"\n    d = np.arange(-180, 181, step)\n    E = V3 / 2 * (1 + np.cos(np.radians(3 * d)))\n    E += gauche_strain * np.exp(-((d - 60) / 32.) ** 2)\n    E += gauche_strain * np.exp(-((d + 60) / 32.) ** 2)\n    E += 2.2 * gauche_strain * np.exp(-(d / 30.) ** 2)\n    E = E - E.min()\n    # steepest descent from start_angle on this surface\n    fine = np.arange(-180, 180.01, 0.5)\n    Ef = np.interp(fine, d, E)\n    i = int(np.argmin(np.abs(fine - start_angle)))\n    while True:\n        lo, hi = max(i - 1, 0), min(i + 1, len(fine) - 1)\n        j = lo if Ef[lo] < Ef[i] else (hi if Ef[hi] < Ef[i] else i)\n        if j == i: break\n        i = j\n    f, ax = _fig()\n    ax.plot(d, E, \"-o\", ms=3.5, color=PRIMARY, label=\"relaxed scan\")\n    ax.axvline(start_angle, color=MUTED, ls=\":\", label=f\"start {start_angle:.0f}\")\n    ax.plot([fine[i]], [Ef[i]], \"v\", ms=12, color=SECOND,\n            label=f\"minimisation stops at {fine[i]:.0f} deg\")\n    ax.set_xlabel(\"C1-C2-C3-C4 dihedral (degrees)\")\n    ax.set_ylabel(\"relative steric energy (kJ/mol)\")\n    ax.set_xticks(range(-180, 181, 60)); ax.legend(fontsize=8)\n    plt.tight_layout(); plt.show()\n    mins = [float(d[k]) for k in range(1, len(d) - 1)\n            if E[k] < E[k - 1] and E[k] < E[k + 1]]\n    return dict(global_min=float(d[E.argmin()]), local_minima=mins,\n                barrier_at_0=float(E[d == 0][0]), stops_at=float(fine[i]))\n\n\ndef search_cost(n_rot=7, step=30):\n    n = int(round(360 / step)) ** n_rot\n    secs = n * 0.05\n    return dict(conformations=n, hours=secs / 3600,\n                years=secs / 3600 / 24 / 365)\n\n\n# ---------------------------------------------------------------- L24\ndef anneal(T=300., n_steps=4000, seed=0):\n    \"\"\"Metropolis walk on a double-well surface: does it escape the local min?\n\n    Global minima sit at +/-180 deg (0 kJ/mol); a local minimum sits at\n    +60 deg (4 kJ/mol) behind a ~26 kJ/mol barrier. At low T the walker is\n    trapped; raising T lets it cross.\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    PLATEAU, DEEP, SHALLOW, W = 30.0, 30.0, 26.0, 26.0\n\n    def E(a):\n        a = np.asarray(a, float)\n        return (PLATEAU\n                - DEEP * np.exp(-((a - 180) / W) ** 2)\n                - DEEP * np.exp(-((a + 180) / W) ** 2)\n                - SHALLOW * np.exp(-((a - 60) / W) ** 2))\n\n    pos = 60.0\n    visited = [pos]\n    kT = max(T, 1.) * 0.0083145                      # kJ/mol per K\n    for _ in range(n_steps):\n        trial = ((pos + rng.normal(0, 20) + 180) % 360) - 180\n        dE = float(E(trial) - E(pos))\n        if dE < 0 or rng.random() < np.exp(-min(dE / kT, 700)):\n            pos = trial\n        visited.append(pos)\n    visited = np.array(visited)\n\n    second_half = visited[len(visited) // 2:]\n    near_global = float(np.mean(np.abs(np.abs(second_half) - 180) < 50) * 100)\n    ever = bool(np.any(np.abs(np.abs(visited) - 180) < 50))\n\n    x = np.linspace(-180, 180, 721)\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))\n    a1.plot(x, E(x), color=PRIMARY, lw=2)\n    a1.set_xlabel(\"dihedral (deg)\"); a1.set_ylabel(\"energy (kJ/mol)\")\n    a1.set_title(\"energy surface\")\n    a1.plot([60], [E(60)], \"o\", color=SECOND, ms=10)\n    a1.annotate(\"start (local, 4 kJ/mol)\", (60, E(60) + 1.5), fontsize=8, ha=\"center\")\n    a1.annotate(\"global (0 kJ/mol)\", (180, E(180) + 1.5), fontsize=8, ha=\"right\")\n    a2.hist(visited, bins=72, color=PRIMARY, alpha=.85)\n    a2.set_xlabel(\"dihedral visited (deg)\"); a2.set_ylabel(\"count\")\n    a2.set_title(f\"T = {T:.0f} K   -   \"\n                 + (\"escaped the local minimum\" if near_global > 15\n                    else \"stayed trapped\"))\n    for ax in (a1, a2):\n        ax.grid(alpha=.25)\n        for sp in (\"top\", \"right\"): ax.spines[sp].set_visible(False)\n    plt.tight_layout(); plt.show()\n    return dict(escaped=near_global > 15, pct_near_global=round(near_global, 1),\n                ever_reached_global=ever, final=round(float(visited[-1]), 1))\n\n\ndef rmsd_overlay(dx=0.0, dy=0.0, rot_deg=0.0):\n    \"\"\"Rigid-body move of a probe onto a template; reports RMSD.\"\"\"\n    tmpl = np.array([[0., 0.], [1.4, 0.3], [2.6, -0.4], [3.9, 0.2], [1.2, 1.7]])\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    probe = (tmpl - tmpl.mean(0)) @ R.T + tmpl.mean(0) + np.array([dx, dy])\n    rmsd = float(np.sqrt(((probe - tmpl) ** 2).sum(1).mean()))\n    f, ax = _fig(6.4, 4.6)\n    ax.plot(*tmpl.T, \"o-\", color=PRIMARY, ms=11, lw=2, label=\"template (rigid active)\")\n    ax.plot(*probe.T, \"s--\", color=SECOND, ms=10, lw=2, label=\"your analogue\")\n    for p, q in zip(tmpl, probe):\n        ax.plot([p[0], q[0]], [p[1], q[1]], color=MUTED, lw=.9, ls=\":\")\n    ax.set_title(f\"RMSD = {rmsd:.2f} A\"); ax.legend(fontsize=8); ax.set_aspect(\"equal\")\n    plt.tight_layout(); plt.show()\n    return dict(rmsd=rmsd, good=rmsd < 0.5)\n\n\n# ---------------------------------------------------------------- L25\n# A snug pocket, 12.0 x 8.0 A, holding a ligand that spans 5.0 A, with a\n# gap in the wall where a polar residue sits. Orientation matters: turned\n# across the pocket the ligand cannot fit without clashing.\n_A, _B = 6.0, 4.1\n_TH = np.linspace(0, 2 * np.pi, 25)[:-1]\n_TH = np.array([t for t in _TH\n                if abs(np.degrees(t) - 90) > 16])          # gap for the residue\nSITE = np.c_[_A * np.cos(_TH), _B * np.sin(_TH)]           # cavity wall atoms\nHBOND_POINT = np.array([0.0, 3.7])                         # polar partner\nLIGAND = np.array([[-2.5, -0.3], [-0.8, 0.3], [0.9, 0.2],\n                   [2.5, -0.4], [0.0, 1.3]])               # last atom is polar\nCLASH_AT, CONTACT_AT, HBOND_AT = 3.0, 3.9, 2.7\n\n\ndef dock_pose(x=0.0, y=0.0, rot_deg=0.0, show=True):\n    \"\"\"Place a rigid 5-atom ligand in a 2-D pocket it very nearly fills.\n\n    Score = -(shape contacts + H-bond) + steric clash penalty, so a MORE\n    NEGATIVE score is a better pose. A pose outside the pocket earns nothing\n    and tends to zero; a pose driven into the wall goes sharply positive.\n    \"\"\"\n    th = np.radians(rot_deg)\n    R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])\n    pose = LIGAND @ R.T + np.array([x, y])\n\n    d = np.linalg.norm(pose[:, None, :] - SITE[None, :, :], axis=2)\n    nearest = d.min(1)\n    bad = nearest < CLASH_AT\n    clash = float(np.sum(np.clip(CLASH_AT - nearest, 0, None) ** 2) * 12)\n    contact = float(np.sum(np.exp(-((nearest - CONTACT_AT) / 1.25) ** 2)) * 2.6)\n    hb_dist = float(np.linalg.norm(pose[4] - HBOND_POINT))\n    hbond = float(6.0 * np.exp(-((hb_dist - HBOND_AT) / 0.95) ** 2))\n    score = -(contact + hbond) + clash\n\n    if show:\n        f, ax = _fig(7.0, 5.4)\n        _full = np.linspace(0, 2 * np.pi, 160)\n        ax.fill(_A * np.cos(_full), _B * np.sin(_full), color=\"#DCE9EF\",\n                zorder=0)\n        ax.plot(np.append(SITE[:, 0], SITE[0, 0]),\n                np.append(SITE[:, 1], SITE[0, 1]), \"o\", color=PRIMARY,\n                ms=15, label=\"binding-site wall\", zorder=1)\n        ax.plot(*HBOND_POINT, \"*\", ms=22, color=\"#B45309\",\n                label=\"polar residue\", zorder=3)\n        chain = pose[:4]\n        ax.plot(chain[:, 0], chain[:, 1], \"-\", color=SECOND, lw=2.6,\n                alpha=.8, zorder=4)\n        ax.plot([chain[1, 0], pose[4, 0]], [chain[1, 1], pose[4, 1]], \"-\",\n                color=SECOND, lw=2.6, alpha=.8, zorder=4)\n        ok = ~bad\n        if ok.any():\n            ax.plot(pose[ok][:, 0], pose[ok][:, 1], \"o\", ms=14,\n                    color=SECOND, label=\"ligand atom\", zorder=5)\n        if bad.any():\n            ax.plot(pose[bad][:, 0], pose[bad][:, 1], \"X\", ms=19,\n                    color=\"#DC2626\", label=\"steric clash\", zorder=6)\n        ax.set_aspect(\"equal\")\n        ax.set_xlim(-9.5, 9.5); ax.set_ylim(-7, 7)\n        ax.legend(fontsize=8, loc=\"lower left\")\n        ax.set_title(f\"score {score:+.1f}    \"\n                     f\"contacts {contact:.1f}   H-bond {hbond:.1f}   \"\n                     f\"clash {clash:.1f}\")\n        plt.tight_layout(); plt.show()\n    return dict(score=round(score, 2), contact=round(contact, 2),\n                hbond=round(hbond, 2), clash=round(clash, 2),\n                clashing_atoms=int(bad.sum()))\n\n\n# ---------------------------------------------------------------- L26\ndef score_vs_truth(cutoff=-7.0, seed=3):\n    \"\"\"Docking scores against crystallographic truth: the ranking problem.\"\"\"\n    rng = np.random.default_rng(seed)\n    n = 60\n    active = rng.random(n) < 0.25\n    rmsd = np.where(active, rng.gamma(2, .9, n), rng.gamma(5, 1.1, n))\n    score = -4.5 - 3.2 * active - rng.normal(0, 1.5, n)\n    picked = score <= cutoff\n    tp = int(np.sum(picked & active)); fp = int(np.sum(picked & ~active))\n    tot_act = int(active.sum())\n    hit = 100 * tp / max(picked.sum(), 1)\n    base = 100 * tot_act / n\n    f, ax = _fig(8.6, 4.6)\n    ax.scatter(score[~active], rmsd[~active], s=46, color=MUTED,\n               label=\"decoy\", alpha=.75)\n    ax.scatter(score[active], rmsd[active], s=56, color=PRIMARY,\n               label=\"true active\", alpha=.9)\n    ax.axvline(cutoff, color=SECOND, lw=2, ls=\"--\",\n               label=f\"your cutoff {cutoff:+.1f}\")\n    ax.axhline(2.0, color=\"#B45309\", lw=1.2, ls=\":\",\n               label=\"2 A - pose counts as correct\")\n    ax.set_xlabel(\"docking score (more negative = better)\")\n    ax.set_ylabel(\"RMSD to crystal pose (A)\")\n    ax.legend(fontsize=8)\n    ax.set_title(f\"picked {int(picked.sum())} compounds: {tp} actives, {fp} decoys\"\n                 f\"   |   hit rate {hit:.0f}% vs {base:.0f}% at random\")\n    plt.tight_layout(); plt.show()\n    return dict(picked=int(picked.sum()), true_actives_found=tp, decoys=fp,\n                total_actives=tot_act, hit_rate=round(hit, 1),\n                random_rate=round(base, 1),\n                enrichment=round(hit / base, 2) if base else 0.0)\n\n\n# ---------------------------------------------------------------- L27\ndef de_novo(h_bond_group=\"amidine\", near_group=\"phenyl\", far_group=\"isopropyl\",\n            linker_bonds=5, fill_site=False):\n    \"\"\"Grow and link fragments into a site with one polar and two greasy pockets.\"\"\"\n    HB = {\"amidine\": 5.0, \"tetrazole\": 4.2, \"phenol\": 3.0, \"methyl\": 0.2,\n          \"isopropyl\": 0.0}\n    GREASE = {\"phenyl\": 3.6, \"cyclohexyl\": 3.9, \"isopropyl\": 2.6,\n              \"methyl\": 1.2, \"amidine\": -1.5, \"tetrazole\": -1.8, \"phenol\": 0.6}\n    SYNTH = {\"amidine\": 1.0, \"tetrazole\": 1.6, \"phenol\": 0.3, \"phenyl\": 0.2,\n             \"cyclohexyl\": 0.5, \"isopropyl\": 0.1, \"methyl\": 0.0}\n    inter = HB.get(h_bond_group, 0) + GREASE.get(near_group, 0) + GREASE.get(far_group, 0)\n    ideal = 6\n    strain = 0.9 * abs(linker_bonds - ideal)\n    synth = SYNTH.get(h_bond_group, 0) + SYNTH.get(near_group, 0) + SYNTH.get(far_group, 0) \\\n        + 0.25 * max(linker_bonds - 8, 0)\n    overfill = 3.0 if fill_site else 0.0\n    total = inter - strain - synth - overfill\n    labels = [\"H-bond region\", \"near pocket\", \"far pocket\",\n              \"linker strain\", \"synthetic cost\", \"over-fill\"]\n    vals = [HB.get(h_bond_group, 0), GREASE.get(near_group, 0),\n            GREASE.get(far_group, 0), -strain, -synth, -overfill]\n    f, ax = _fig(9.2, 3.8)\n    cols = [PRIMARY if v >= 0 else SECOND for v in vals]\n    ax.barh(labels, vals, color=cols)\n    ax.axvline(0, color=\"k\", lw=.9)\n    ax.set_xlabel(\"contribution to design score\")\n    ax.set_title(f\"design score {total:+.1f}\"\n                 + (\"   (over-filled: no room left to optimise)\" if fill_site else \"\"))\n    plt.tight_layout(); plt.show()\n    return dict(score=round(total, 2), interactions=round(inter, 2),\n                strain=round(strain, 2), synthetic_cost=round(synth, 2),\n                overfill_penalty=overfill)\n\n\n# ---------------------------------------------------------------- L29 / L30\nCOUMARIN = dict(\n    X=[\"H\", \"Me\", \"OH\", \"OMe\", \"OCF3\", \"NH2\", \"F\", \"Cl\", \"CF3\", \"CN\", \"NO2\"],\n    pIC50=np.array([8.36, 8.38, 8.01, 8.44, 7.94, 7.46, 8.55, 8.48, 8.24, 7.97, 8.59]),\n    pi=np.array([0., .49, -.67, -.09, 1.02, -1.23, .14, .71, .88, -.57, -.26]),\n    Vdw=np.array([.08, 1.01, .49, 1.49, 1.60, .67, .36, 1.07, 1.11, 1.14, 1.20]),\n    F=np.array([0., -.04, .28, .25, .37, .02, .42, .40, .37, .50, .66]))\n\n_PI = {\"H\": 0., \"Me\": .56, \"Cl\": .71, \"NO2\": -.28, \"OMe\": -.02, \"NH2\": -1.23, \"F\": .14}\n_MR = {\"H\": .103, \"Me\": .565, \"Cl\": .603, \"NO2\": .736, \"OMe\": .787, \"NH2\": .542, \"F\": .092}\n_F = {\"H\": 0., \"Me\": .01, \"Cl\": .42, \"NO2\": .65, \"OMe\": .29, \"NH2\": .08, \"F\": .45}\n_R = {\"H\": 0., \"Me\": -.18, \"Cl\": -.19, \"NO2\": .13, \"OMe\": -.56, \"NH2\": -.74, \"F\": -.39}\n_TAC = [(\"tacrine\", \"H\", \"H\", 6.59), (\"I\", \"Me\", \"H\", 5.09), (\"II\", \"H\", \"Me\", 7.00),\n        (\"III\", \"Cl\", \"H\", 6.26), (\"IV\", \"H\", \"Cl\", 8.00), (\"V\", \"NO2\", \"H\", 5.52),\n        (\"VI\", \"H\", \"NO2\", 7.55), (\"VII\", \"H\", \"OMe\", 6.45), (\"VIII\", \"NH2\", \"H\", 5.42),\n        (\"IX\", \"H\", \"F\", 7.06), (\"X\", \"Cl\", \"Cl\", 6.32), (\"XI\", \"OMe\", \"OMe\", 5.29)]\n\ndef tacrine_table():\n    rows = []\n    for cid, X, Y, p in _TAC:\n        rows.append(dict(cpd=cid, X=X, Y=Y, pIC50=p, piX=_PI[X], piY=_PI[Y],\n                         MRX=_MR[X], MRY=_MR[Y], F_XY=_F[X] + _F[Y], R_XY=_R[X] + _R[Y]))\n    return rows\n\n\ndef _lstsq(X, y):\n    X = np.column_stack([np.asarray(X, float).reshape(len(y), -1), np.ones(len(y))])\n    beta, *_ = np.linalg.lstsq(X, y, rcond=None)\n    pred = X @ beta\n    ss_res = float(((y - pred) ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())\n    r2 = 1 - ss_res / ss_tot\n    n, p = len(y), X.shape[1]\n    s = float(np.sqrt(ss_res / (n - p))) if n > p else float(\"nan\")\n    return beta, pred, r2, s\n\n\ndef qsar_single(param=\"pi\", quadratic=False):\n    \"\"\"One descriptor at a time on the coumarin MAO-B series.\"\"\"\n    y = COUMARIN[\"pIC50\"]; x = COUMARIN[param]\n    cols = np.column_stack([x, x ** 2]) if quadratic else x\n    beta, pred, r2, s = _lstsq(cols, y)\n    f, ax = _fig(7.6, 4.4)\n    ax.scatter(x, y, s=70, color=PRIMARY, zorder=3)\n    for xi, yi, t in zip(x, y, COUMARIN[\"X\"]):\n        ax.annotate(t, (xi, yi), fontsize=8, xytext=(4, 4), textcoords=\"offset points\")\n    xs = np.linspace(x.min(), x.max(), 200)\n    ys = beta[0] * xs + beta[1] * xs ** 2 + beta[2] if quadratic else beta[0] * xs + beta[1]\n    ax.plot(xs, ys, color=SECOND, lw=2,\n            label=(\"quadratic\" if quadratic else \"linear\") + f\"   r = {np.sqrt(max(r2,0)):.3f}   s = {s:.3f}\")\n    ax.set_xlabel({\"pi\": \"substituent hydrophobicity  pi\",\n                   \"Vdw\": \"van der Waals volume\", \"F\": \"inductive factor  F\"}[param])\n    ax.set_ylabel(\"pIC50 (MAO-B)\"); ax.legend(fontsize=9)\n    plt.tight_layout(); plt.show()\n    return dict(param=param, quadratic=quadratic,\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3))\n\n\ndef qsar_multi(use=(\"MRX\",), show=True):\n    \"\"\"Add descriptors and watch r rise while leave-one-out gets worse.\"\"\"\n    rows = tacrine_table()\n    y = np.array([r[\"pIC50\"] for r in rows])\n    X = np.column_stack([[r[c] for r in rows] for c in use])\n    beta, pred, r2, s = _lstsq(X, y)\n    loo = []\n    for i in range(len(y)):\n        m = np.ones(len(y), bool); m[i] = False\n        b, _, _, _ = _lstsq(X[m], y[m])\n        loo.append(float(np.dot(np.append(X[i], 1.0), b)))\n    loo = np.array(loo)\n    q2 = 1 - float(((y - loo) ** 2).sum()) / float(((y - y.mean()) ** 2).sum())\n    if show:\n        f, ax = _fig(5.6, 5.2)\n        ax.scatter(pred, y, s=72, color=PRIMARY, label=\"fitted\", zorder=3)\n        ax.scatter(loo, y, s=52, color=SECOND, marker=\"s\",\n                   label=\"left-out prediction\", alpha=.85, zorder=3)\n        lim = [min(y.min(), pred.min(), loo.min()) - .4,\n               max(y.max(), pred.max(), loo.max()) + .4]\n        ax.plot(lim, lim, \"k--\", lw=1); ax.set_xlim(lim); ax.set_ylim(lim)\n        ax.set_xlabel(\"predicted pIC50\"); ax.set_ylabel(\"observed pIC50\")\n        ax.set_title(\" + \".join(use) + f\"\\nr = {np.sqrt(max(r2,0)):.3f}    \"\n                     f\"s = {s:.3f}    q2 = {q2:.3f}\")\n        ax.legend(fontsize=8); plt.tight_layout(); plt.show()\n    return dict(descriptors=list(use), n=len(y), k=len(use),\n                r=round(float(np.sqrt(max(r2, 0))), 3), s=round(s, 3),\n                q2=round(q2, 3), compounds_per_parameter=round(len(y) / len(use), 1))\n\n\n# ---------------------------------------------------------------- L31\nBIOISOSTERE = {\n    \"carboxylic acid (COOH)\": dict(pKa=4.2, logP=0.0, MW=45, HBD=1, HBA=2, note=\"parent\"),\n    \"tetrazole\":              dict(pKa=4.9, logP=0.5, MW=69, HBD=1, HBA=4, note=\"classic acid mimic\"),\n    \"acylsulfonamide\":        dict(pKa=4.5, logP=0.6, MW=121, HBD=1, HBA=4, note=\"acid mimic, bulkier\"),\n    \"2,6-difluorophenol\":     dict(pKa=7.1, logP=1.9, MW=130, HBD=1, HBA=1, note=\"lipophilic acid mimic\"),\n    \"hydroxamic acid\":        dict(pKa=9.0, logP=-0.4, MW=61, HBD=2, HBA=3, note=\"weaker acid, metal binder\"),\n    \"primary amide (CONH2)\":  dict(pKa=None, logP=-1.5, MW=44, HBD=2, HBA=2, note=\"neutral\"),\n    \"methyl ester\":           dict(pKa=None, logP=0.6, MW=59, HBD=0, HBA=2, note=\"neutral, hydrolysable\"),\n}\n\ndef bioisostere_swap(replacement=\"tetrazole\", need_acidic=True, need_permeable=True):\n    p = BIOISOSTERE[\"carboxylic acid (COOH)\"]; r = BIOISOSTERE[replacement]\n    keeps_acid = r[\"pKa\"] is not None and r[\"pKa\"] < 6.5\n    more_lipo = r[\"logP\"] > p[\"logP\"]\n    verdict = []\n    if need_acidic:\n        verdict.append((\"keeps the ionised acid at pH 7.4\", keeps_acid))\n    if need_permeable:\n        verdict.append((\"raises lipophilicity for permeability\", more_lipo))\n    verdict.append((\"molecular weight increase under 60 Da\", r[\"MW\"] - p[\"MW\"] < 60))\n    f, ax = _fig(8.8, 3.6)\n    labels = [\"pKa\", \"logP\", \"MW (Da)\", \"HBD\", \"HBA\"]\n    par = [p[\"pKa\"] if p[\"pKa\"] else 0, p[\"logP\"], p[\"MW\"] / 20, p[\"HBD\"], p[\"HBA\"]]\n    rep = [r[\"pKa\"] if r[\"pKa\"] else 0, r[\"logP\"], r[\"MW\"] / 20, r[\"HBD\"], r[\"HBA\"]]\n    i = np.arange(len(labels)); w = .38\n    ax.bar(i - w / 2, par, w, label=\"carboxylic acid (parent)\", color=MUTED)\n    ax.bar(i + w / 2, rep, w, label=replacement, color=PRIMARY)\n    ax.set_xticks(i); ax.set_xticklabels(labels)\n    ax.set_title(f\"{replacement} - {r['note']}   (MW shown /20)\")\n    ax.legend(fontsize=8); ax.axhline(0, color=\"k\", lw=.8)\n    plt.tight_layout(); plt.show()\n    for text, ok in verdict:\n        print((\"  MET      \" if ok else \"  NOT MET  \") + text)\n    return dict(replacement=replacement, pKa=r[\"pKa\"], logP=r[\"logP\"],\n                dMW=r[\"MW\"] - p[\"MW\"], requirements_met=sum(1 for _, o in verdict if o),\n                requirements_total=len(verdict))\n\n\n# ---------------------------------------------------------------- L32\ndef prodrug_kinetics(promoiety=\"ethyl ester\", esterase_activity=1.0, hours=8.0):\n    \"\"\"First-order activation: prodrug -> active drug, and patient variability.\"\"\"\n    K = {\"ethyl ester\": 0.45, \"5-indanyl ester (EWG)\": 1.60,\n         \"pivaloyl ester (hindered)\": 0.10, \"amide\": 0.02, \"no promoiety\": 0.0}\n    k = K[promoiety] * max(esterase_activity, 0.0)\n    t = np.linspace(0, hours, 400)\n    pro = np.exp(-k * t); act = 1 - pro\n    f, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.3))\n    a1.plot(t, pro * 100, color=MUTED, lw=2, label=\"prodrug (inactive)\")\n    a1.plot(t, act * 100, color=PRIMARY, lw=2.4, label=\"active drug\")\n    a1.set_xlabel(\"hours\"); a1.set_ylabel(\"% of dose\"); a1.legend(fontsize=8)\n    a1.set_title(f\"{promoiety}   k = {k:.2f} /h\")\n    for lab, mult, col in [(\"low enzyme (x0.3)\", .3, SECOND),\n                           (\"typical (x1)\", 1., PRIMARY),\n                           (\"high enzyme (x3)\", 3., \"#15803D\")]:\n        a2.plot(t, (1 - np.exp(-K[promoiety] * mult * t)) * 100, lw=2.2,\n                color=col, label=lab)\n    a2.set_xlabel(\"hours\"); a2.set_ylabel(\"% converted to active drug\")\n    a2.set_title(\"same prodrug, three patients\"); a2.legend(fontsize=8)\n    for ax in (a1, a2):\n        ax.grid(alpha=.25); ax.set_ylim(-3, 103)\n        for s in (\"top\", \"right\"): ax.spines[s].set_visible(False)\n    plt.tight_layout(); plt.show()\n    half = float(np.log(2) / k) if k > 0 else float(\"inf\")\n    at2 = float((1 - np.exp(-k * 2)) * 100)\n    spread = float(abs((1 - np.exp(-K[promoiety] * 3 * 2)) -\n                       (1 - np.exp(-K[promoiety] * .3 * 2))) * 100)\n    return dict(promoiety=promoiety, k_per_h=round(k, 3),\n                half_life_h=(round(half, 2) if np.isfinite(half) else None),\n                pct_active_at_2h=round(at2, 1),\n                patient_spread_at_2h_pct=round(spread, 1))\n\n\n# ---------------------------------------------------------------- L22 extra\ndef method_choice(atoms=20, want=\"partial charges and an electrostatic map\"):\n    \"\"\"Which level of theory answers the question at an affordable cost.\"\"\"\n    needs_electrons = want in (\"partial charges and an electrostatic map\",\n                               \"heat of formation\",\n                               \"transition state of a reaction\")\n    if not needs_electrons:\n        rec = \"Molecular mechanics\"\n        why = (\"The question is about geometry and sterics only. A force field \"\n               \"treats atoms as spheres on springs and never computes an \"\n               \"electron distribution, which is all the speed comes from.\")\n    elif atoms <= 30:\n        rec = \"Ab initio quantum mechanics\"\n        why = (\"Electron distribution is required and the molecule is small \"\n               \"enough that a first-principles calculation is affordable.\")\n    elif atoms <= 150:\n        rec = \"Semi-empirical quantum mechanics\"\n        why = (\"Electron distribution is required but the molecule is too large \"\n               \"for ab initio, so experimentally fitted parameters are used to \"\n               \"simplify the integrals.\")\n    else:\n        rec = \"Molecular mechanics, and reconsider the question\"\n        why = (\"No quantum method is affordable at this size. Either cut the \"\n               \"molecule down to the fragment that matters, or accept that \"\n               \"electronic properties are out of reach.\")\n    cost = {\"Molecular mechanics\": 1, \"Semi-empirical quantum mechanics\": 1e3,\n            \"Ab initio quantum mechanics\": 1e6}\n    f, ax = _fig(8.6, 2.9)\n    names = list(cost); vals = [cost[n] for n in names]\n    cols = [PRIMARY if n in rec else MUTED for n in names]\n    ax.barh(names, vals, color=cols)\n    ax.set_xscale(\"log\"); ax.set_xlabel(\"relative cost (log scale)\")\n    ax.set_title(f\"{atoms} atoms   -   {want}\")\n    plt.tight_layout(); plt.show()\n    print(\"RECOMMEND:\", rec); print()\n    print(_tw(why))\n    return dict(atoms=atoms, question=want, recommend=rec,\n                needs_electron_distribution=needs_electrons)\n\n\n# ---------------------------------------------------------------- L25 extra\ndef site_surface(probe_radius=1.4, show_probe=True):\n    \"\"\"van der Waals surface versus the solvent-accessible (Connolly) surface.\n\n    Two gate atoms are placed so the opening between their van der Waals\n    surfaces is exactly GAP angstroms. A spherical probe passes only if its\n    radius does not exceed GAP/2, so the narration below is derived from the\n    coordinates rather than asserted.\n    \"\"\"\n    R = 1.7\n    GAP = 1.60                       # surface-to-surface opening, angstroms\n    d = 2 * R + GAP                  # required centre-to-centre separation\n    gate = [(-d / 2, 0.0, R), (d / 2, 0.0, R)]\n    wall = [(-5.6, -1.0, R), (-5.0, -3.0, R), (-3.4, -4.4, R),\n            (-1.2, -5.0, R), (1.2, -5.0, R), (3.4, -4.4, R),\n            (5.0, -3.0, R), (5.6, -1.0, R)]\n    atoms = gate + wall\n    half_gap = GAP / 2\n    fits = probe_radius <= half_gap\n\n    gx, gy = np.meshgrid(np.linspace(-8, 8, 460), np.linspace(-7.5, 4.5, 360))\n    vdw = np.zeros_like(gx, bool); excluded = np.zeros_like(gx, bool)\n    for (cx, cy, r) in atoms:\n        dd = (gx - cx) ** 2 + (gy - cy) ** 2\n        vdw |= dd <= r ** 2\n        excluded |= dd <= (r + probe_radius) ** 2\n\n    f, ax = _fig(7.4, 6.0)\n    ax.contourf(gx, gy, excluded.astype(float), levels=[.5, 1.5],\n                colors=[\"#CBD5E1\"], alpha=.8)\n    ax.contour(gx, gy, excluded.astype(float), levels=[.5], colors=[SECOND],\n               linewidths=2.2)\n    ax.contourf(gx, gy, vdw.astype(float), levels=[.5, 1.5], colors=[MUTED],\n                alpha=.95)\n    ax.contour(gx, gy, vdw.astype(float), levels=[.5], colors=[\"#334155\"],\n               linewidths=1.3)\n    if show_probe:\n        ax.add_patch(plt.Circle((0.0, 0.0), probe_radius, fill=False,\n                                ec=PRIMARY, lw=2.4, ls=\"--\"))\n        ax.plot([0], [0], \"x\", color=PRIMARY, ms=9)\n    ax.annotate(\"\", xy=(-d / 2 + R, 0), xytext=(d / 2 - R, 0),\n                arrowprops=dict(arrowstyle=\"<->\", color=PRIMARY, lw=1.6))\n    ax.text(0, .45, f\"{GAP:.2f} A opening\", ha=\"center\", color=PRIMARY,\n            fontsize=9)\n    ax.set_aspect(\"equal\"); ax.set_xlim(-8, 8); ax.set_ylim(-7.5, 4.5)\n    ax.set_xticks([]); ax.set_yticks([])\n    ax.set_title(f\"probe radius {probe_radius:.2f} A\\n\"\n                 \"dark grey = atoms (van der Waals)    \"\n                 \"orange outline = surface this probe can reach\")\n    plt.tight_layout(); plt.show()\n\n    print(f\"opening between the gate atoms : {GAP:.2f} A\")\n    print(f\"largest probe that passes      : {half_gap:.2f} A radius\")\n    print(f\"this probe                     : {probe_radius:.2f} A radius\")\n    print()\n    print(_tw(\n        (\"The probe fits through the opening, so the cavity behind the gate is \"\n         \"mapped as accessible and a ligand atom may be placed there.\"\n         if fits else\n         \"The probe is too large to pass, so the opening is sealed over. The \"\n         \"cavity behind it never appears on the docking surface, which is the \"\n         \"point: no ligand atom could reach it either.\")))\n    return dict(probe_radius=probe_radius, opening_angstrom=GAP,\n                max_probe_radius_that_fits=half_gap,\n                probe_passes=bool(fits),\n                excluded_volume_grid_points=int(excluded.sum()))\n\n\n# ------------------------------------------------- L31, real SwissBioisostere\ndef bioisostere_real(parent=\"carboxylic acid\", replacement=\"primary amide\",\n                     need_not_worse=0.80, max_mw_increase=60.0, show=True):\n    \"\"\"Judge a replacement against real matched-pair data.\n\n    Data: SwissBioisostere (CC-BY 4.0), ChEMBL 36. See swissbio.py for the\n    full citation and for what was changed on retrieval.\n    \"\"\"\n    import swissbio as sb\n    rows = {r[\"name\"]: r for r in sb.table(parent)}\n    if replacement not in rows:\n        print(f\"'{replacement}' is not in the curated {parent} list.\")\n        print(\"Available:\", \", \".join(sorted(rows)[:12]), \"...\")\n        return None\n    r = rows[replacement]\n    n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n    not_worse = (r[\"better\"] + r[\"similar\"]) / n\n    total = sb.QUERIES[parent][\"total\"]\n\n    if show:\n        f, (a1, a2) = plt.subplots(1, 2, figsize=(12.6, 4.2))\n        # how activity changed across the real pairs\n        a1.barh([\"worse\", \"similar\", \"better\"],\n                [r[\"worse\"], r[\"similar\"], r[\"better\"]],\n                color=[SECOND, MUTED, \"#15803D\"])\n        a1.set_xlabel(f\"matched molecular pairs (n = {n})\")\n        a1.set_title(f\"{replacement}\\nreplacing {parent} in real compounds\")\n        for s in (\"top\", \"right\"): a1.spines[s].set_visible(False)\n\n        # where this replacement sits against the others by frequency\n        tab = sb.table(parent)\n        ranks = [x[\"rank\"] for x in tab]; freqs = [x[\"freq\"] for x in tab]\n        cols = [PRIMARY if x[\"name\"] == replacement else \"#CBD5E1\" for x in tab]\n        a2.bar(ranks, freqs, color=cols)\n        a2.set_xlabel(f\"rank by frequency (top {len(tab)} of {total:,})\")\n        a2.set_ylabel(\"matched pairs\")\n        a2.set_title(f\"this replacement is rank {r['rank']} of {total:,}\")\n        for s in (\"top\", \"right\"): a2.spines[s].set_visible(False)\n        plt.tight_layout(); plt.show()\n\n    print(f\"SMILES              {r['smiles']}\")\n    print(f\"frequency           {r['freq']:,} matched pairs  (rank {r['rank']} of {total:,})\")\n    print(f\"activity outcome    {r['better']} better / {r['similar']} similar / {r['worse']} worse\")\n    print(f\"not worse           {not_worse*100:.0f}% of pairs\")\n    print(f\"mean change         logP {r['dlogp']:+.2f}   tPSA {r['dtpsa']:+.1f}   MW {r['dmw']:+.1f}\")\n    print()\n    checks = [(f\"at least {need_not_worse*100:.0f}% of pairs not worse\", not_worse >= need_not_worse),\n              (f\"molecular weight increase under {max_mw_increase:.0f} Da\",\n               r[\"dmw\"] < max_mw_increase)]\n    for text, okd in checks:\n        print((\"  MET      \" if okd else \"  NOT MET  \") + text)\n    return dict(name=replacement, smiles=r[\"smiles\"], freq=r[\"freq\"], rank=r[\"rank\"],\n                not_worse=round(not_worse, 3), dlogp=r[\"dlogp\"], dtpsa=r[\"dtpsa\"],\n                dmw=r[\"dmw\"], met=sum(1 for _, o in checks if o), total_checks=len(checks))\n\n\ndef bioisostere_ranking(parent=\"carboxylic acid\", min_freq=300, show=True):\n    \"\"\"Frequency is not success: rank the common replacements by outcome.\"\"\"\n    import swissbio as sb\n    tab = [r for r in sb.table(parent) if r[\"freq\"] >= min_freq]\n    for r in tab:\n        n = r[\"better\"] + r[\"similar\"] + r[\"worse\"]\n        r[\"not_worse\"] = (r[\"better\"] + r[\"similar\"]) / n if n else 0\n    by_freq = sorted(tab, key=lambda r: -r[\"freq\"])\n    by_ok = sorted(tab, key=lambda r: -r[\"not_worse\"])\n    if show:\n        f, ax = _fig(9.6, 5.0)\n        ax.scatter([r[\"freq\"] for r in tab], [r[\"not_worse\"] * 100 for r in tab],\n                   s=70, color=PRIMARY, zorder=3)\n        for r in tab:\n            if r[\"freq\"] > 600 or r[\"not_worse\"] > 0.88:\n                ax.annotate(r[\"name\"], (r[\"freq\"], r[\"not_worse\"] * 100),\n                            fontsize=8, xytext=(5, 4), textcoords=\"offset points\")\n        ax.set_xscale(\"log\")\n        ax.set_xlabel(f\"how often it was done (matched pairs, log scale)\")\n        ax.set_ylabel(\"% of pairs where activity was not worse\")\n        ax.set_title(f\"replacing {parent}: popularity against outcome\")\n        plt.tight_layout(); plt.show()\n    print(f\"{'most common':<30}{'most reliable':<30}\")\n    print(\"-\" * 60)\n    for i in range(min(6, len(tab))):\n        print(f\"{by_freq[i]['name'][:28]:<30}{by_ok[i]['name'][:28]:<30}\")\n    return dict(n=len(tab),\n                most_common=by_freq[0][\"name\"], most_reliable=by_ok[0][\"name\"],\n                same=by_freq[0][\"name\"] == by_ok[0][\"name\"])\n", "sims.py", "exec"), _sm.__dict__)
sys.modules["sims"] = _sm
sims = _sm

ITEMS = [
    Item("P1", "In a carrier-linked prodrug, what does the promoiety do?",
         {"A": "It protects the drug from the immune system during transport", "B": "It binds the target directly once the drug has been absorbed", "C": "It changes the physical properties and is later cleaved off", "D": "It provides the pharmacological activity until the drug acts"},
         "C", {"B": "The promoiety is removed before activity; it does not bind the target.", "D": "A carrier-linked prodrug is inactive until the promoiety is cleaved.", "A": "Immune evasion is not the role a promoiety plays.", "_correct": "The promoiety is attached to alter properties such as solubility, stability or taste, and is removed in the body to release the active drug."},
         bloom="Understand", silo="", section="precheck"),
    Item("P2", "An ester prodrug is activated faster when the alcohol carries an electron-withdrawing aryl group. Why?",
         {"A": "The compound becomes more water soluble and so hydrolyses", "B": "The ester bond becomes longer and therefore weaker overall", "C": "The aryl group attracts esterase enzymes to the molecule", "D": "The carbonyl becomes more electrophilic and easier to attack"},
         "D", {"B": "Bond length changes negligibly and is not the operative factor.", "C": "Esterases are not recruited by an aryl group in this way.", "A": "Greater solubility does not itself accelerate ester hydrolysis.", "_correct": "Withdrawing electron density from the carbonyl carbon makes it more electrophilic, so nucleophilic attack and hydrolysis proceed faster."},
         bloom="Understand", silo="", section="precheck"),
    Item("P3", "A prodrug behaves consistently in plasma but gives variable exposure across patients. What does that suggest?",
         {"A": "Activation depends on an enzyme whose activity differs between people", "B": "The prodrug is chemically unstable and decomposes before it is taken", "C": "The assay used to measure plasma concentrations is poorly calibrated", "D": "The drug itself has a narrow therapeutic window in these patients"},
         "A", {"B": "Chemical instability would show up in the plasma incubations as well.", "C": "A miscalibrated assay would distort the plasma results too.", "D": "A narrow window makes variability dangerous but does not cause it.", "_correct": "Consistency in plasma rules out chemical instability, so the variability must come from enzymes or gut flora that differ between individuals."},
         bloom="Understand", silo="", section="precheck"),
    Item("C1", "The ethyl ester showed a far larger between-patient spread at two hours than the indanyl ester did. Why?",
         {"A": "The ethyl ester is chemically unstable and decomposes unpredictably", "B": "At an intermediate rate, two hours falls where the curves differ most", "C": "Esterase activity varies more widely for ethyl esters than aryl ones", "D": "The indanyl ester is activated by a route that needs no enzyme at all"},
         "B", {"C": "The same threefold range of enzyme activity was applied to both compounds.", "D": "Both are hydrolysed enzymatically; only the rate constant differs between them.", "A": "Both are modelled as clean first-order hydrolysis with no decomposition.", "_correct": "The indanyl ester is nearly complete in every patient by two hours, so the curves converge, while an intermediate rate leaves the three patients at very different extents."},
         bloom="Analyze", silo="4", section="concept"),
    Item("C2", "You must choose a promoiety for a drug with a narrow therapeutic window. Which consideration should dominate?",
         {"A": "Reliability of activation across patients, not raw speed alone", "B": "The lowest possible rate constant, to avoid any sudden release", "C": "The highest possible rate constant, to minimise exposure time", "D": "The lowest molecular weight promoiety that can be attached"},
         "A", {"B": "A very slow promoiety may never deliver a therapeutic concentration at all.", "C": "Speed alone ignores whether every patient achieves the same exposure.", "D": "Promoiety mass is a minor consideration beside unpredictable activation.", "_correct": "With a narrow window the danger is between-patient variability in exposure, so a promoiety whose activation is consistent matters more than one that is merely fast."},
         bloom="Evaluate", silo="4", section="concept"),
    Item("Q1", "SAR data: OCH2 active, NHCH2 active, SCH2 inactive, CH2CH2 inactive, CH=CH inactive. What does the pattern establish about the oxygen?",
         {"A": "It contributes bulk that the binding site sterically requires", "B": "It stabilizes the active conformation through its bond angle", "C": "It raises lipophilicity enough to drive membrane permeation", "D": "It acts as a hydrogen-bond acceptor at that position in the site"},
         "D", {"A": "is ruled out by the similarly sized CH2CH2", "C": "is ruled out by the direction of pi values", "B": "is not distinguished by this series", "_correct": "Only O and NH can accept a hydrogen bond at that position, and the three groups that cannot accept one are exactly the inactive analogues."},
         bloom="Analyze", silo="4", section="post"),
    Item("Q2", "A CF3 group is introduced two bonds from a basic amine that must engage an anionic site residue. Potency falls. Why?",
         {"A": "CF3 sterically blocks the amine's approach to the site residue", "B": "CF3 lowers the amine's pKa, reducing the protonated fraction", "C": "CF3 is metabolized to an acid that competes for the same site", "D": "CF3 raises lipophilicity, which excludes the drug from plasma"},
         "B", {"A": "is possible but not the chapter's mechanism at that distance", "C": "is wrong because CF3 is metabolically robust", "D": "is unrelated to the binding step", "_correct": "A strongly electron-withdrawing CF3 lowers the amine's basicity, so less of it is protonated and available for the ionic interaction."},
         bloom="Apply", silo="4", section="post"),
    Item("Q3", "Two esters of one drug differ only in the alcohol: ethanol versus 5-indanol. How will activation compare, and why?",
         {"A": "The indanyl ester hydrolyzes faster, as the aryl group withdraws density", "B": "Neither hydrolyzes in plasma, since aryl esters require liver enzymes", "C": "Both hydrolyze identically, since esterases are insensitive to structure", "D": "The ethyl ester hydrolyzes faster, as smaller esters are less hindered"},
         "A", {"C": "is false", "B": "contradicts plasma esterase activity", "D": "applies a sterics-only argument", "_correct": "The indanyl group is aryl and electron-withdrawing, which makes the ester carbonyl more electrophilic and speeds hydrolysis."},
         bloom="Apply", silo="4", section="post"),
    Item("Q4", "Chloramphenicol is formulated as the palmitate ester for oral suspension and the succinate ester for injection. What does this contrast demonstrate?",
         {"A": "Two promoieties are required before any prodrug can be marketed", "B": "Ester prodrugs invariably increase the aqueous solubility of a drug", "C": "Promoiety choice is dictated by the route and the dosage form needed", "D": "The parent drug is inactive until both esters have been hydrolyzed"},
         "C", {"B": "is contradicted by the palmitate", "D": "misstates the parent's activity", "A": "is an invented requirement", "_correct": "The palmitate is poorly soluble and tasteless, suiting an oral suspension, while the water-soluble succinate suits an injection."},
         bloom="Analyze", silo="4", section="post"),
    Item("Q5", "A prodrug performs consistently in plasma incubations but gives highly variable exposure across patients. Which explanation fits best?",
         {"A": "The promoiety itself is pharmacologically active in some patients", "B": "Activating enzyme or gut flora activity differs between individuals", "C": "The linkage hydrolyzes chemically rather than enzymatically in vivo", "D": "Plasma incubation overestimates the drug's intrinsic potency in vivo"},
         "B", {"A": "would not produce variable parent exposure", "C": "is wrong because chemical hydrolysis would be more uniform", "D": "confuses potency with exposure", "_correct": "Plasma incubation tests only chemical and plasma-esterase hydrolysis, so between-patient variability must come from enzymes or gut flora that differ."},
         bloom="Evaluate", silo="4", section="post"),
]

gb = Gradebook("L32", "Prodrugs")
gb.add_many(ITEMS)
print("Ready - 10 questions loaded. Scroll down and begin.")

# ===== BEGIN structure-drawing and calculator kit (lecture notes) =====
# ---------------------------------------------------------------------------
# PHS 511 notebook helper library (pure Python + matplotlib; no RDKit, no network)
# Draws textbook-oriented 2D skeletal structures from explicit coordinates so
# every figure has a fixed, reviewed orientation:
#   * acyl chains: C1 (carbonyl) on the LEFT, omega (methyl) end on the RIGHT
#   * glycerol / sphingosine backbones vertical (sn-1 top, sn-3 bottom)
#   * steroids: rings A-B-C-D left to right, C3-OH lower left, C18/C19 up
# ---------------------------------------------------------------------------
import math
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, FancyArrowPatch, FancyBboxPatch, Circle

COL = {
    "ink": "#1f2430", "muted": "#6b7280",
    "ester": "#1d6fb8", "amide": "#c2410c", "ether": "#7c3aed",
    "phos": "#15803d", "oh": "#b91c1c", "dbl": "#d97706",
    "delta": "#1d6fb8", "omega": "#b91c1c", "drug": "#be123c",
    "polar": "#0e7490", "nonpolar": "#a16207", "hl": "#fde68a",
}
S30, C30 = 0.5, math.cos(math.radians(30))
UNIT_IN = 0.36   # inches per bond length -> identical scale in every figure


class Mol:
    """Atoms with explicit 2D coordinates; bonds with order/style/highlight."""

    def __init__(self, name=""):
        self.name, self.atoms, self.bonds, self.notes, self.boxes = name, [], [], [], []

    def add(self, x, y, label="", color=None, ha="center", size=None, anchor=None):
        """anchor: index of the label character that sits on the atom (default: first C/N/O/P/S)."""
        self.atoms.append(dict(x=x, y=y, label=label, color=color, ha=ha, size=size, anchor=anchor))
        return len(self.atoms) - 1

    def bond(self, i, j, order=1, style="plain", hl=None, color=None, side=None):
        self.bonds.append(dict(i=i, j=j, order=order, style=style, hl=hl, color=color, side=side))
        return len(self.bonds) - 1

    def note(self, x, y, text, color=None, size=8.5, ha="center", va="center", weight="normal"):
        self.notes.append(dict(x=x, y=y, text=text, color=color, size=size, ha=ha, va=va, weight=weight))

    def pos(self, i):
        a = self.atoms[i]
        return a["x"], a["y"]



def _turtle(m, start, n, d0, turns, orders, hl=None):
    idx = [start]
    x, y = m.pos(start)
    d = d0
    for k in range(1, n + 1):
        x, y = x + math.cos(math.radians(d)), y + math.sin(math.radians(d))
        idx.append(m.add(x, y))
        m.bond(idx[-2], idx[-1], order=orders.get(k, 1), hl=hl)
        if k < n:
            d += turns[k]
    return idx


def chain(m, start, n, heading=0.0, first=+1, cis=(), trans=(), hl=None, flip=()):
    """Zigzag chain of n carbons grown from atom `start` (start = carbon 0).
    Bond k joins chain carbons k-1 and k. The turn at carbon k normally
    alternates sign (zigzag). If bond k is a cis C=C, the turn at carbon k
    repeats the previous sign so both substituents sit on the same side.
    `flip` lists carbons where a single bond is drawn rotated (same-sign turn)
    to steer a long chain back into its row; it does not change the molecule."""
    turns, orders = {}, {}
    sign = -first
    for k in range(1, n):
        if k >= 2:
            sign = sign if (k in cis or k in flip) else -sign
        turns[k] = 60 * sign
    for k in list(cis) + list(trans):
        orders[k] = 2
    return _turtle(m, start, n, heading + 30 * first, turns, orders, hl)


# ---------------------------------------------------------------------------
# Rendering
# ---------------------------------------------------------------------------
SUBS = set("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089\u207a\u207b\u00b9\u00b2\u00b3\u2074")
EM = 1.0 / (72 * UNIT_IN)   # data units per point at the fixed drawing scale


def _cw(ch):
    """Approximate glyph advance (em) for DejaVu Sans."""
    if ch in SUBS:
        return 0.40
    if ch in "()":
        return 0.39
    if ch.isupper():
        return 0.72
    if ch.isdigit():
        return 0.64
    return 0.56


def _label_box(lab, fs, anchor=None):
    """(left, right, half-height) of a label in data units, measured from the
    anchor glyph centre (first C/N/O/P/S), which sits exactly on the atom."""
    em = fs * EM
    ws = [_cw(c) * em for c in lab]
    k = anchor if anchor is not None else next((i for i, c in enumerate(lab) if c in "CNOPS"), 0)
    left = sum(ws[:k]) + ws[k] / 2
    right = sum(ws[k + 1:]) + ws[k] / 2
    return left, right, 0.36 * fs * EM * 1.0


def _label_radius(lab, ux, uy, fs=10.5, anchor=None):
    """Distance from atom centre to the label box edge along (ux, uy), plus a gap."""
    if not lab:
        return 0.0
    left, right, hh = _label_box(lab, fs, anchor)
    tx = (right if ux > 0 else left) / abs(ux) if abs(ux) > 1e-9 else 1e9
    ty = hh / abs(uy) if abs(uy) > 1e-9 else 1e9
    return min(tx, ty) + 0.08


def draw(ax, m, fs=10.5):
    A = m.atoms
    nb = {i: [] for i in range(len(A))}
    for b in m.bonds:
        nb[b["i"]].append(b["j"]); nb[b["j"]].append(b["i"])
    # highlights first (under everything)
    for b in m.bonds:
        if b["hl"]:
            (x1, y1), (x2, y2) = m.pos(b["i"]), m.pos(b["j"])
            ax.plot([x1, x2], [y1, y2], color=b["hl"], lw=9, alpha=0.30,
                    solid_capstyle="round", zorder=1)
    for b in m.bonds:
        i, j = b["i"], b["j"]
        (x1, y1), (x2, y2) = m.pos(i), m.pos(j)
        L = math.hypot(x2 - x1, y2 - y1)
        ux, uy = (x2 - x1) / L, (y2 - y1) / L
        r1 = _label_radius(A[i]["label"], ux, uy, A[i]["size"] or fs, A[i]["anchor"])
        r2 = _label_radius(A[j]["label"], ux, uy, A[j]["size"] or fs, A[j]["anchor"])
        ax1, ay1 = x1 + ux * r1, y1 + uy * r1
        ax2, ay2 = x2 - ux * r2, y2 - uy * r2
        c = b["color"] or COL["ink"]
        st = b["style"]
        if st == "wedge":
            px, py = -uy * 0.11, ux * 0.11
            ax.add_patch(Polygon([(ax1, ay1), (ax2 + px, ay2 + py), (ax2 - px, ay2 - py)],
                                 closed=True, fc=c, ec=c, lw=0.6, zorder=3))
            continue
        if st == "hash":
            nseg = 7
            for s in range(nseg + 1):
                f = s / nseg
                cx, cy = ax1 + (ax2 - ax1) * f, ay1 + (ay2 - ay1) * f
                w = 0.02 + 0.10 * f
                ax.plot([cx - uy * w, cx + uy * w], [cy + ux * w, cy - ux * w],
                        color=c, lw=1.1, zorder=3, solid_capstyle="butt")
            continue
        if st == "wavy":
            n = 40
            xs = [ax1 + (ax2 - ax1) * k / n - uy * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ys = [ay1 + (ay2 - ay1) * k / n + ux * 0.06 * math.sin(k / n * 6 * math.pi) for k in range(n + 1)]
            ax.plot(xs, ys, color=c, lw=1.4, zorder=3)
            continue
        px, py = -uy, ux
        if b["order"] == 1:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round",
                    ls="--" if st == "dashed" else "-")
            continue
        side = b["side"]
        if side is None:
            if A[i]["label"] or A[j]["label"]:
                side = 0                      # C=O, C=N+: centred pair
            else:
                sx = sy = 0.0
                for k in nb[i]:
                    if k != j:
                        sx += A[k]["x"] - x1; sy += A[k]["y"] - y1
                for k in nb[j]:
                    if k != i:
                        sx += A[k]["x"] - x2; sy += A[k]["y"] - y2
                cross = ux * sy - uy * sx
                side = 1 if cross >= -1e-6 else -1
        if side == 0:
            for sgn in (+1, -1):
                o = 0.075 * sgn
                ax.plot([ax1 + px * o, ax2 + px * o], [ay1 + py * o, ay2 + py * o],
                        color=c, lw=1.5, zorder=3, solid_capstyle="round")
        else:
            ax.plot([ax1, ax2], [ay1, ay2], color=c, lw=1.6, zorder=3, solid_capstyle="round")
            o, sh = 0.15 * side, 0.16
            ax.plot([ax1 + px * o + ux * sh, ax2 + px * o - ux * sh],
                    [ay1 + py * o + uy * sh, ay2 + py * o - uy * sh],
                    color=c, lw=1.5, zorder=3, solid_capstyle="round")
    for a in A:
        if a["label"]:
            l_, r_, _ = _label_box(a["label"], a["size"] or fs, a["anchor"])
            ax.text(a["x"] + (r_ - l_) / 2, a["y"], a["label"], ha="center", va="center",
                    fontsize=a["size"] or fs, color=a["color"] or COL["ink"],
                    zorder=5, fontweight="normal")
    for n_ in m.notes:
        ax.text(n_["x"], n_["y"], n_["text"], ha=n_["ha"], va=n_["va"], fontsize=n_["size"],
                color=n_["color"] or COL["muted"], zorder=6, fontweight=n_["weight"])


def extent(m, pad=0.7):
    xs = [a["x"] for a in m.atoms] + [n["x"] for n in m.notes]
    ys = [a["y"] for a in m.atoms] + [n["y"] for n in m.notes]
    return min(xs) - pad, max(xs) + pad, min(ys) - pad, max(ys) + pad




def show(m, title=None, unit=UNIT_IN, extra=None, caption=None, show_now=True):
    """One structure at the fixed scale (UNIT_IN inches per bond)."""
    x0, x1, y0, y1 = extent(m)
    if extra and hasattr(extra, "bounds"):
        bx0, bx1, by0, by1 = extra.bounds
        x0, x1, y0, y1 = min(x0, bx0), max(x1, bx1), min(y0, by0), max(y1, by1)
    th = 0.42 if title else 0.0
    ch = 0.40 if caption else 0.0
    W, H = (x1 - x0) * unit, (y1 - y0) * unit
    fig = plt.figure(figsize=(W, H + th + ch))
    ax = fig.add_axes([0, ch / (H + th + ch), 1, H / (H + th + ch)])
    draw(ax, m)
    if extra:
        extra(ax)
    ax.set_xlim(x0, x1); ax.set_ylim(y0, y1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        fig.text(0.0, 1 - (th / 2) / (H + th + ch), title, fontsize=11.5, fontweight="bold",
                 color=COL["ink"], va="center", ha="left")
    if caption:
        fig.text(0.0, (ch / 2) / (H + th + ch), caption, fontsize=9, color=COL["muted"],
                 va="center", ha="left")
    if show_now:
        plt.show()
    return fig


def grid(mols, titles, ncols=2, unit=UNIT_IN, align="left", suptitle=None, captions=None, show_now=True, bottom=0.0):
    """Panels drawn at one common scale; each panel box has the same width and
    the row height of its tallest member, so structures line up cleanly."""
    exts = [extent(m) for m in mols]
    W = max(max(e[1] - e[0] for e in exts), 3.6 / unit)   # panels at least 3.6 in wide
    nrows = math.ceil(len(mols) / ncols)
    import textwrap as _tw
    maxch = max(12, int(W * unit * 72 / (10.5 * 0.60)) - 1)
    wrapped = [_tw.fill(t, maxch) for t in titles]
    rowH = []
    for r in range(nrows):
        rowH.append(max(exts[k][3] - exts[k][2] for k in range(r * ncols, min(len(mols), (r + 1) * ncols))))
    row_th = []
    for r in range(nrows):
        nl = max(wrapped[k].count("\n") + 1 for k in range(r * ncols, min(len(mols), (r + 1) * ncols)))
        row_th.append(0.27 + 0.19 * nl)
    fig_w = W * unit * ncols
    fig_h = sum(h * unit + th for h, th in zip(rowH, row_th)) + (0.5 if suptitle else 0) + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    top = 1.0 - (0.5 / fig_h if suptitle else 0)
    y_cursor = top
    for r in range(nrows):
        h_frac = (rowH[r] * unit) / fig_h
        t_frac = row_th[r] / fig_h
        y_cursor -= (h_frac + t_frac)
        for c in range(ncols):
            k = r * ncols + c
            if k >= len(mols):
                break
            ax = fig.add_axes([c / ncols, y_cursor, 1 / ncols, h_frac])
            m = mols[k]
            x0, x1, y0, y1 = exts[k]
            draw(ax, m)
            if align == "left":
                ax.set_xlim(x0, x0 + W)
            else:
                cx = (x0 + x1) / 2
                ax.set_xlim(cx - W / 2, cx + W / 2)
            cy = (y0 + y1) / 2
            ax.set_ylim(cy - rowH[r] / 2, cy + rowH[r] / 2)
            ax.set_aspect("equal"); ax.axis("off")
            ax.text(0.01, 1.0 + 0.02, wrapped[k], transform=ax.transAxes, fontsize=10.5, linespacing=1.15,
                    fontweight="bold", color=COL["ink"], va="bottom", ha="left")
    if suptitle:
        fig.text(0.005, 1 - 0.25 / fig_h, suptitle, fontsize=12.5, fontweight="bold", va="center")
    if show_now:
        plt.show()
    return fig


# ---------------------------------------------------------------------------
# Structure builders (coordinates chosen to match textbook orientation)
# ---------------------------------------------------------------------------
def _outer(m, i, nbrs, dist=0.55):
    """Point on the open (outer) side of atom i, away from its neighbours."""
    x, y = m.pos(i)
    sx = sum(m.atoms[k]["x"] - x for k in nbrs); sy = sum(m.atoms[k]["y"] - y for k in nbrs)
    L = math.hypot(sx, sy) or 1.0
    return x - sx / L * dist, y - sy / L * dist


def fatty_acid(n, cis=(), trans=(), name="", numbers=None, ionized=False, origin=(0.0, 0.0),
               hl_double=True, flip=()):
    """Free fatty acid, C1 (COOH) on the LEFT, methyl (omega) end on the RIGHT.
    cis/trans: Delta positions (bond between C_p and C_p+1).
    numbers: None | 'delta' | 'omega' | 'both'"""
    m = Mol(name)
    ox, oy = origin
    c1 = m.add(ox, oy)
    o_dbl = m.add(ox, oy + 1.0, "O")
    o_h = m.add(ox - C30, oy - 0.5, "⁻O" if ionized else "HO")
    m.bond(c1, o_dbl, 2); m.bond(c1, o_h, 1)
    hl = COL["hl"] if hl_double else None
    ids = chain(m, c1, n - 1, heading=0, first=-1, cis=cis, trans=trans, flip=flip)
    if hl_double:
        for b in m.bonds:
            if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
                b["hl"] = COL["dbl"]
    m.carbons = ids  # carbons[0] = C1
    if numbers:
        dbs = sorted(list(cis) + list(trans))
        last = dbs[-1] if dbs else n - 2
        for p, a in enumerate(ids, start=1):
            if p == 1:
                nb = [o_dbl, o_h, ids[1]]
            elif p == n:
                nb = [ids[p - 2]]
            else:
                nb = [ids[p - 2], ids[p]]
            if numbers in ("delta", "both"):
                x, y = _outer(m, a, nb, 0.50)
                m.note(x, y, str(p), color=COL["delta"], size=7.5)
            w = n - p + 1
            if numbers == "omega" or (numbers == "both" and p >= last):
                if p == n:
                    x, y = _outer(m, a, nb, 0.50)
                    y -= 0.42 if numbers == "both" else 0.0
                else:
                    x, y = _outer(m, a, nb, 0.50 if numbers == "omega" else -0.62)
                m.note(x, y, ("\u03c9" + str(w)) if w == 1 else str(w), color=COL["omega"], size=7.5,
                       weight="bold")
    return m


def acyl_row(m, o_atom, n, cis=(), trans=(), carbonyl_up=True, flip=None):
    """Acyl group C(=O)-chain attached to ester/amide atom o_atom, chain to the right."""
    x, y = m.pos(o_atom)
    c1 = m.add(x + C30, y + (0.5 if carbonyl_up else -0.5))
    m.bond(o_atom, c1, 1)
    od = m.add(x + C30, y + (1.5 if carbonyl_up else -1.5), "O")
    m.bond(c1, od, 2)
    if flip is None:   # keep long chains in their row: rotate the bond 3 carbons past the last cis C=C
        flip = tuple(p + 2 for p in cis)
    ids = chain(m, c1, n - 1, heading=0, first=-1 if carbonyl_up else +1, cis=cis, trans=trans, flip=flip)
    for b in m.bonds:
        if b["order"] == 2 and not m.atoms[b["i"]]["label"] and not m.atoms[b["j"]]["label"]:
            b["hl"] = COL["dbl"]
    return c1, od, ids


ROW = 2.6  # vertical spacing of backbone carbons in E-shaped lipids


def backbone(m, labels, x=0.0, y0=0.0):
    """Vertical 3-carbon backbone (glycerol or sphingosine C1-C3), top->bottom."""
    ids = []
    for k, lab in enumerate(labels):
        ids.append(m.add(x, y0 - k * ROW, lab))
        if k:
            m.bond(ids[k - 1], ids[k], 1)
    return ids


def ester(m, c_backbone, n, cis=(), trans=(), hl=True, cut=None):
    x, y = m.pos(c_backbone)
    o = m.add(x + 1.15, y, "O")
    b0 = m.bond(c_backbone, o, 1, hl=COL["ester"] if hl else None)
    c1, od, ids = acyl_row(m, o, n, cis, trans)
    m.bonds[-(len(ids)) - 1]["hl"] = None
    # highlight O-C(=O) (the ester bond lipases cut)
    for b in m.bonds:
        if b["i"] == o and b["j"] == c1:
            b["hl"] = COL["ester"] if hl else None
            if cut:
                b["color"] = COL["drug"]
    return o, c1, ids


def triacylglycerol(acyls=((16, ()), (18, (9,)), (18, (9, 12))), name="Triacylglycerol",
                    sn=True, cuts=None):
    """E-shaped TAG; glycerol vertical, sn-1 top. acyls: ((n, cis-positions), ...)."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    rows = []
    for k, (n, cis) in enumerate(acyls):
        rows.append(ester(m, g[k], n, cis=cis))
    if sn:
        for k in range(3):
            x, y = m.pos(g[k])
            m.note(x - 0.95, y, "sn-%d" % (k + 1), color=COL["muted"], size=8.5, ha="right")
    m.g, m.rows = g, rows
    return m


def mark_cut(m, o_idx, c_idx, text, color=None, dy=-0.62, dx=0.0):
    x1, y1 = m.pos(o_idx); x2, y2 = m.pos(c_idx)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my - 0.05, "✂", color=color or COL["drug"], size=13)
    m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5, ha="left" if dx else "center")


def phosphocholine(m, c_attach, head="choline"):
    """-O-P(=O)(O-)-O-CH2-CH2-N+(CH3)3 to the right of c_attach (horizontal)."""
    x, y = m.pos(c_attach)
    o1 = m.add(x + 1.15, y, "O"); m.bond(c_attach, o1, 1)
    p = m.add(x + 2.3, y, "P", color=COL["phos"]); m.bond(o1, p, 1, hl=COL["phos"])
    po = m.add(x + 2.3, y + 1.05, "O"); m.bond(p, po, 2)
    pm = m.add(x + 2.3, y - 1.05, "O⁻"); m.bond(p, pm, 1)
    o2 = m.add(x + 3.45, y, "O"); m.bond(p, o2, 1, hl=COL["phos"])
    if head == "choline":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "N⁺(CH₃)₃"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.6, y - 1.0, "choline", color=COL["muted"], size=8)
    elif head == "ethanolamine":
        c_a = m.add(x + 3.45 + C30, y + 0.5); m.bond(o2, c_a)
        c_b = m.add(x + 3.45 + 2 * C30, y); m.bond(c_a, c_b)
        nn = m.add(x + 3.45 + 3 * C30, y + 0.5, "NH₃⁺"); m.bond(c_b, nn)
        m.note(x + 3.45 + 1.4, y - 1.0, "ethanolamine", color=COL["muted"], size=8)
    return p


def phosphatidylcholine(name="Phosphatidylcholine (lecithin)", a1=(16, ()), a2=(18, (9,))):
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    ester(m, g[0], *a1); ester(m, g[1], *a2)
    phosphocholine(m, g[2], "choline")
    for k in range(3):
        x, y = m.pos(g[k]); m.note(x - 0.95, y, "sn-%d" % (k + 1), size=8.5, ha="right")
    m.g = g
    return m


def plasmalogen(name="Plasmalogen (ethanolamine)", n1=16, a2=(18, (9,))):
    """sn-1 vinyl ether: CH2-O-CH=CH-R (cis), no carbonyl."""
    m = Mol(name)
    g = backbone(m, ["CH₂", "CH", "CH₂"])
    x, y = m.pos(g[0])
    o = m.add(x + 1.15, y, "O"); m.bond(g[0], o, 1, hl=COL["ether"])
    ids = chain(m, o, n1, heading=0, first=+1, cis=(2,), flip=(4,))
    # bond O-C1' is ether; C1'=C2' vinyl (cis, as in natural plasmalogens)
    for b in m.bonds:
        if b["i"] == o and b["j"] == ids[1]:
            b["hl"] = COL["ether"]
        if b["order"] == 2:
            b["hl"] = COL["ether"]
    ester(m, g[1], *a2)
    phosphocholine(m, g[2], "ethanolamine")
    for k in range(3):
        xx, yy = m.pos(g[k]); m.note(xx - 0.95, yy, "sn-%d" % (k + 1), size=8.5, ha="right")
    return m


def sphingo_core(m, fa_n=18, fa_cis=()):
    """Sphingosine drawn as the same vertical 3-carbon backbone:
    C3 top (CH-OH + trans CH=CH tail), C2 middle (CH-NH-acyl, amide), C1 bottom (CH2-O-head)."""
    s = backbone(m, ["CH", "CH", "CH₂"])
    x, y = m.pos(s[0])
    oh = m.add(x - 1.15, y, "HO"); m.bond(s[0], oh, 1, hl=COL["oh"])
    ids = chain(m, s[0], 15, heading=0, first=+1, trans=(1,))
    # C4=C5 trans: chain bond 1 is C3-C4? -> rebuild explicitly
    m.bonds[[k for k, b in enumerate(m.bonds) if b["i"] == s[0] and b["j"] == ids[1]][0]]["order"] = 1
    for b in m.bonds:
        if b["i"] == ids[1] and b["j"] == ids[2]:
            b["order"] = 2; b["hl"] = COL["dbl"]
    # amide at C2
    x2, y2 = m.pos(s[1])
    nh = m.add(x2 + 1.4, y2, "NH"); m.bond(s[1], nh, 1)
    c1, od, acyl = acyl_row(m, nh, fa_n, cis=fa_cis)
    for b in m.bonds:
        if b["i"] == nh and b["j"] == c1:
            b["hl"] = COL["amide"]
    for k, lab in enumerate(["C3", "C2", "C1"]):
        xx, yy = m.pos(s[k]); m.note(xx - 0.25, yy - 0.55, lab, size=7.5)
    return s


def sphingomyelin(name="Sphingomyelin"):
    m = Mol(name)
    s = sphingo_core(m, 18)
    phosphocholine(m, s[2], "choline")
    return m


def galactosylceramide(name="Galactosylceramide (a cerebroside)"):
    m = Mol(name)
    s = sphingo_core(m, 24, fa_cis=(15,))
    x, y = m.pos(s[2])
    o = m.add(x + 1.15, y, "O"); m.bond(s[2], o, 1, hl=COL["polar"])
    # beta-D-galactopyranose, Haworth-free chair-less hexagon (ring O upper right)
    cx, cy, r = x + 3.0, y - 0.1, 1.0
    ang = [180, 120, 60, 0, -60, -120]
    pts = [m.add(cx + r * math.cos(math.radians(a)), cy + r * math.sin(math.radians(a)),
                 "O" if k == 1 else "") for k, a in enumerate(ang)]
    for k in range(6):
        m.bond(pts[k], pts[(k + 1) % 6])
    m.bond(o, pts[0], 1, hl=COL["polar"])
    m.note(cx, cy - 0.05, "Gal", color=COL["polar"], size=9, weight="bold")
    return m


# --- steroids --------------------------------------------------------------
def _hex(cx, cy):
    return [(cx + math.cos(math.radians(a)), cy + math.sin(math.radians(a))) for a in (90, 30, -30, -90, -150, 150)]


def steroid_core(m, ox=0.0, oy=0.0, dbl=()):
    """Gonane nucleus in standard orientation. Returns dict C1..C17 -> atom index.
    Unit hexagon geometry with vertical shared A/B edge (C5-C10)."""
    r3 = math.sqrt(3)
    P = {}
    A = _hex(ox, oy)                      # top, ur, lr, bottom, ll, ul
    P.update({1: A[0], 10: A[1], 5: A[2], 4: A[3], 3: A[4], 2: A[5]})
    B = _hex(ox + r3, oy)
    P.update({9: B[0], 8: B[1], 6: B[3], 7: B[2]})
    C = _hex(ox + 1.5 * r3, oy + 1.5)
    P.update({12: C[0], 13: C[1], 14: C[2], 11: C[5]})
    # ring D: regular pentagon on edge C13-C14, outward (to the right)
    (x13, y13), (x14, y14) = P[13], P[14]
    mx, my = (x13 + x14) / 2, (y13 + y14) / 2
    s = 1.0
    apo = s / (2 * math.tan(math.radians(36)))
    cx, cy = mx + apo, my
    R = s / (2 * math.sin(math.radians(36)))
    base = math.degrees(math.atan2(y13 - cy, x13 - cx))
    pent = [(cx + R * math.cos(math.radians(base - 72 * k)), cy + R * math.sin(math.radians(base - 72 * k)))
            for k in range(5)]
    # pent[0]=C13, going clockwise: C17, C16, C15, C14
    P.update({17: pent[1], 16: pent[2], 15: pent[3]})
    idx = {k: m.add(v[0], v[1]) for k, v in P.items()}
    ring_bonds = [(1, 2), (2, 3), (3, 4), (4, 5), (5, 10), (10, 1), (5, 6), (6, 7), (7, 8), (8, 9), (9, 10),
                  (9, 11), (11, 12), (12, 13), (13, 14), (14, 8), (13, 17), (17, 16), (16, 15), (15, 14)]
    for a, b in ring_bonds:
        m.bond(idx[a], idx[b], 2 if (a, b) in dbl or (b, a) in dbl else 1)
    m.st = idx
    return idx


def subst(m, at, dx, dy, label="", style="plain", order=1, hl=None, color=None):
    x, y = m.pos(at)
    k = m.add(x + dx, y + dy, label, color=color)
    m.bond(at, k, order, style=style, hl=hl)
    return k


def side_chain_cholestane(m, st, extra_ethyl=False):
    """C20-C27 side chain on C17 (drawn up-right, zigzag)."""
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 4, heading=0, first=-1)
    c25 = ids[-1]
    x25, y25 = m.pos(c25)
    c26 = m.add(x25 + C30, y25 + 0.5); m.bond(c25, c26)
    c27 = m.add(x25, y25 - 1.0); m.bond(c25, c27)
    if extra_ethyl:
        x24, y24 = m.pos(ids[3])
        ex, ey = _outer(m, ids[3], [ids[2], ids[4]], 1.0)
        e1 = m.add(ex, ey); m.bond(ids[3], e1, hl=COL["nonpolar"])
        e2 = m.add(ex + C30, ey + (0.5 if ey > y24 else -0.5)); m.bond(e1, e2, hl=COL["nonpolar"])
    return ids


def cholesterol(name="Cholesterol", sitosterol=False):
    m = Mol(name)
    st = steroid_core(m, dbl=((5, 6),))
    ho = subst(m, st[3], -C30, -0.5, "HO", style="wedge", hl=COL["oh"])
    subst(m, st[10], 0.0, 0.95, "", style="wedge")     # C19 methyl
    subst(m, st[13], 0.0, 0.95, "", style="wedge")     # C18 methyl
    side_chain_cholestane(m, st, extra_ethyl=sitosterol)
    m.note(m.pos(st[3])[0] - 0.2, m.pos(st[3])[1] + 0.45, "3β", size=7.5)
    for k, lab in ((1, "A"), (2, "B"), (3, "C")):
        pass
    r3 = math.sqrt(3)
    for (cx, cy, lab) in ((0, 0, "A"), (r3, 0, "B"), (1.5 * r3, 1.5, "C")):
        m.note(cx, cy, lab, color="#9aa3b2", size=9, weight="bold")
    x13, y13 = m.pos(st[13]); x14, y14 = m.pos(st[14])
    m.note((x13 + x14) / 2 + 0.68, (y13 + y14) / 2, "D", color="#9aa3b2", size=9, weight="bold")
    m.st = st
    return m


def bile_acid(name="Glycocholate", oh=(("3", "a"), ("7", "a"), ("12", "a")), conj="glycine", ionized=True):
    """5-beta bile acid; oh entries (position, 'a' alpha=hash | 'b' beta=wedge)."""
    m = Mol(name)
    st = steroid_core(m)
    subst(m, st[10], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[13], 0.0, 0.95, "", style="wedge", hl=COL["nonpolar"])
    subst(m, st[5], 0.0, -0.85, "H", style="wedge")      # 5-beta H (A/B cis)
    vec = {"3": (-C30, -0.5), "7": (C30 * 0.0, -1.0), "12": (0.0, 1.0)}
    for pos, face in oh:
        dx, dy = vec[pos]
        if pos == "12":
            dx, dy = (-0.35, 0.9)
        if pos == "7":
            dx, dy = (0.35, -0.95)
        subst(m, st[int(pos)], dx, dy, "HO" if pos == "3" else "OH",
              style="hash" if face == "a" else "wedge", hl=COL["polar"])
    x, y = m.pos(st[17])
    c20 = m.add(x + 0.30, y + 0.95); m.bond(st[17], c20, 1, style="wedge")
    c21 = m.add(x - 0.45, y + 1.6); m.bond(c20, c21)
    ids = chain(m, c20, 3, heading=0, first=-1)
    c24 = ids[-1]
    x24, y24 = m.pos(c24)
    o24 = m.add(x24, y24 + 1.0, "O"); m.bond(c24, o24, 2)
    if conj == "glycine":
        nh = m.add(x24 + C30, y24 - 0.5, "NH"); m.bond(c24, nh, 1, hl=COL["amide"])
        ca = m.add(x24 + 2 * C30, y24); m.bond(nh, ca)
        cc = m.add(x24 + 3 * C30, y24 - 0.5); m.bond(ca, cc)
        m.bond(cc, m.add(x24 + 3 * C30, y24 - 1.5, "O"), 2)
        m.bond(cc, m.add(x24 + 4 * C30, y24, "O⁻" if ionized else "OH"), 1, hl=COL["polar"])
    else:
        m.bond(c24, m.add(x24 + C30, y24 - 0.5, "OH"), 1, hl=COL["polar"])
    m.st = st
    return m


# ---------------------------------------------------------------------------
# General 2D sketch helpers: rings, fused rings, substituents, reaction schemes
# (built on draw.py; every coordinate is explicit so orientation is fixed)
# ---------------------------------------------------------------------------
def _nbrs(m, i):
    out = []
    for b in m.bonds:
        if b["i"] == i:
            out.append(b["j"])
        elif b["j"] == i:
            out.append(b["i"])
    return out


def bond_between(m, i, j):
    for b in m.bonds:
        if (b["i"], b["j"]) in ((i, j), (j, i)):
            return b
    return None


def ring(m, cx, cy, n=6, rot=90.0, labels=None, side=1.0):
    """Regular n-ring with bond length `side`, first vertex at angle `rot` (deg),
    vertices counter-clockwise. labels: list of n labels ('' = carbon).
    Returns atom indices in ring order."""
    R = side / (2 * math.sin(math.pi / n))
    ids = []
    for k in range(n):
        a = math.radians(rot + 360.0 * k / n)
        ids.append(m.add(cx + R * math.cos(a), cy + R * math.sin(a), (labels[k] if labels else "")))
    for k in range(n):
        m.bond(ids[k], ids[(k + 1) % n], 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def fuse(m, a, b, n=6, labels=None, side=1.0):
    """New regular n-ring sharing edge a-b, built on the side away from the
    existing neighbours. labels: for the n-2 NEW atoms, in order from b onward.
    Returns ring atoms [a, b, new...]."""
    (xa, ya), (xb, yb) = m.pos(a), m.pos(b)
    mx, my = (xa + xb) / 2, (ya + yb) / 2
    ex, ey = xb - xa, yb - ya
    L = math.hypot(ex, ey)
    px, py = -ey / L, ex / L
    # decide which side is free: away from centroid of current neighbours
    nb = [k for k in _nbrs(m, a) + _nbrs(m, b) if k not in (a, b)]
    if nb:
        cx_ = sum(m.atoms[k]["x"] for k in nb) / len(nb); cy_ = sum(m.atoms[k]["y"] for k in nb) / len(nb)
        if (cx_ - mx) * px + (cy_ - my) * py > 0:
            px, py = -px, -py
    apo = side / (2 * math.tan(math.pi / n))
    cx, cy = mx + px * apo, my + py * apo
    R = side / (2 * math.sin(math.pi / n))
    # walk from b around the new ring (away from a)
    ang_b = math.atan2(yb - cy, xb - cx)
    ang_a = math.atan2(ya - cy, xa - cx)
    step = 2 * math.pi / n
    # direction: going from b, the next vertex must not be a
    d = (ang_b - ang_a + math.pi) % (2 * math.pi) - math.pi
    sgn = 1 if d > 0 else -1
    ids = [a, b]
    prev = b
    for k in range(1, n - 1):
        ang = ang_b + sgn * step * k
        lab = labels[k - 1] if labels else ""
        i = m.add(cx + R * math.cos(ang), cy + R * math.sin(ang), lab)
        m.bond(prev, i, 1)
        ids.append(i); prev = i
    m.bond(prev, a, 1)
    m.rings = getattr(m, "rings", []) + [ids]
    return ids


def set_double(m, pairs):
    """pairs: iterable of (i, j) atom-index pairs to make double bonds."""
    for i, j in pairs:
        b = bond_between(m, i, j)
        if b is None:
            raise ValueError("no bond %d-%d" % (i, j))
        b["order"] = 2


def kekule(m, ids, start=0):
    """Alternate double bonds around ring `ids` beginning with bond start->start+1."""
    n = len(ids)
    for k in range(start, start + n, 2):
        bond_between(m, ids[k % n], ids[(k + 1) % n])["order"] = 2


def outward(m, i, dist=1.0):
    """Unit outward direction from atom i (away from its neighbours)."""
    x, y = m.pos(i)
    nb = _nbrs(m, i)
    sx = sum(m.atoms[k]["x"] - x for k in nb); sy = sum(m.atoms[k]["y"] - y for k in nb)
    L = math.hypot(sx, sy)
    if L < 1e-9:
        return (dist, 0.0)
    return (-sx / L * dist, -sy / L * dist)


def sub(m, i, label="", angle=None, dist=1.0, order=1, style="plain", hl=None, color=None, anchor=None):
    """Attach a substituent to atom i: outward by default, or at an absolute angle (deg)."""
    x, y = m.pos(i)
    if angle is None:
        dx, dy = outward(m, i, dist)
    else:
        dx, dy = dist * math.cos(math.radians(angle)), dist * math.sin(math.radians(angle))
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, style=style, hl=hl)
    return k


def grow(m, i, n, angle=None, first=+1, cis=(), trans=(), flip=()):
    """Zigzag chain of n atoms from atom i; heading = outward unless angle given."""
    if angle is None:
        dx, dy = outward(m, i)
        angle = math.degrees(math.atan2(dy, dx))
    return chain(m, i, n, heading=angle, first=first, cis=cis, trans=trans, flip=flip)


def label(m, i, text, anchor=None):
    m.atoms[i]["label"] = text
    m.atoms[i]["anchor"] = anchor


def hl_bonds(m, pairs, color):
    for i, j in pairs:
        bond_between(m, i, j)["hl"] = color


def hl_atoms_note(m, i, text, color=None, dist=0.75, size=8):
    x, y = m.pos(i)
    dx, dy = outward(m, i, dist)
    m.note(x + dx, y + dy, text, color=color or COL["muted"], size=size)


def shift(m, dx, dy):
    for a in m.atoms:
        a["x"] += dx; a["y"] += dy
    for n in m.notes:
        n["x"] += dx; n["y"] += dy
    return m


def rotate(m, deg, about=None):
    """Rotate a whole drawing (use to fix an orientation, e.g. to put a ring flat)."""
    if about is None:
        xs = [a["x"] for a in m.atoms]; ys = [a["y"] for a in m.atoms]
        about = ((min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2)
    c, s = math.cos(math.radians(deg)), math.sin(math.radians(deg))
    for obj in m.atoms + m.notes:
        x, y = obj["x"] - about[0], obj["y"] - about[1]
        obj["x"], obj["y"] = about[0] + c * x - s * y, about[1] + s * x + c * y
    return m


def mirror(m, axis="x"):
    """Mirror left-right (axis='x') or top-bottom ('y'); wedges keep their sense."""
    for obj in m.atoms + m.notes:
        if axis == "x":
            obj["x"] = -obj["x"]
        else:
            obj["y"] = -obj["y"]
    return m


def merge(dst, src, dx=0.0, dy=0.0):
    """Copy all atoms/bonds/notes of src into dst with an offset; returns index map."""
    base = len(dst.atoms)
    for a in src.atoms:
        b = dict(a); b["x"] += dx; b["y"] += dy
        dst.atoms.append(b)
    for b in src.bonds:
        c = dict(b); c["i"] += base; c["j"] += base
        dst.bonds.append(c)
    for n in src.notes:
        c = dict(n); c["x"] += dx; c["y"] += dy
        dst.notes.append(c)
    return base


# ---------------------------------------------------------------------------
# Reaction / pathway scheme: molecules in a row (wrapping), arrows with labels
# ---------------------------------------------------------------------------
def scheme(items, title=None, per_row=3, unit=UNIT_IN, gap=3.2, caption=None, show_now=True):
    """items: [Mol, ("enzyme", "cofactor note"), Mol, (...), Mol ...]
    Molecules are drawn at one scale; arrows carry an enzyme label above and
    an optional cofactor/regulation note below. Rows wrap after per_row molecules."""
    mols = [x for x in items if isinstance(x, Mol)]
    arrows = [x for x in items if not isinstance(x, Mol)]
    exts = [extent(mm, pad=0.4) for mm in mols]
    rows = [list(range(k, min(k + per_row, len(mols)))) for k in range(0, len(mols), per_row)]
    row_h = [max(exts[k][3] - exts[k][2] for k in r) for r in rows]
    row_w = [sum(exts[k][1] - exts[k][0] for k in r) + gap * (len(r) - 1) for r in rows]
    W = max(row_w) + (1.6 if len(rows) > 1 else 0)
    H = sum(row_h) + 1.6 * (len(rows) - 1)
    th = 0.45 if title else 0.0
    ch = 0.4 if caption else 0.0
    fig = plt.figure(figsize=(W * unit, H * unit + th + ch))
    ax = fig.add_axes([0, ch / (H * unit + th + ch), 1, H * unit / (H * unit + th + ch)])
    ax.set_xlim(0, W); ax.set_ylim(-H, 0); ax.set_aspect("equal"); ax.axis("off")
    y_top = 0.0
    pos = {}
    for r, ids in enumerate(rows):
        x = 0.0 if r == 0 else 1.6
        cy = y_top - row_h[r] / 2
        for k in ids:
            x0, x1, y0, y1 = exts[k]
            dx = x - x0; dy = cy - (y0 + y1) / 2
            mm = Mol(); merge(mm, mols[k], dx, dy)
            draw(ax, mm)
            pos[k] = (x, x + (x1 - x0), cy, y1 - y0)
            x += (x1 - x0) + gap
        y_top -= row_h[r] + 1.6
    for k, lab in enumerate(arrows):
        if k + 1 >= len(mols):
            break
        enz, note = (lab, "") if isinstance(lab, str) else (lab[0], lab[1] if len(lab) > 1 else "")
        xa0, xa1, ya, ha = pos[k]; xb0, xb1, yb, hb = pos[k + 1]
        same = abs(ya - yb) < 1e-6
        if same:
            p, q = (xa1 + 0.25, ya), (xb0 - 0.25, yb)
        else:  # wrap: from under the last molecule of a row to above the first of the next
            p, q = ((xa0 + xa1) / 2, ya - ha / 2 + 0.1), ((xb0 + xb1) / 2, yb + hb / 2 - 0.1)
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.4, color=COL["ink"],
                                     connectionstyle="arc3,rad=0" if same else "arc3,rad=-0.12"))
        mx, my = (p[0] + q[0]) / 2, (p[1] + q[1]) / 2
        box = dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.9)
        if same:
            ax.text(mx, my + 0.3, enz, ha="center", va="bottom", fontsize=8.5, color=COL["ester"],
                    fontweight="bold", bbox=box)
            if note:
                ax.text(mx, my - 0.3, note, ha="center", va="top", fontsize=8, color=COL["muted"], bbox=box)
        else:
            ax.text(mx, my, enz + ("\n" + note if note else ""), ha="center", va="center", fontsize=8.5,
                    color=COL["ester"], bbox=box)
    if title:
        fig.text(0.0, 1 - (th / 2) / (H * unit + th + ch), title, fontsize=11.5, fontweight="bold", va="center")
    if caption:
        fig.text(0.0, (ch / 2) / (H * unit + th + ch), caption, fontsize=9, color=COL["muted"], va="center")
    if show_now:
        plt.show()
    return fig


try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(s): return s


def _md(lines):
    display(Markdown("\n".join(lines) if isinstance(lines, (list, tuple)) else lines))


# ---------------------------------------------------------------------------
# L32 lecture-notes figures and calculators: prodrugs
# Every prodrug is drawn directly ABOVE its active drug on one template, so the
# shared core sits in the same place and only the promoiety changes.
# Blue halo = promoiety; scissors = the bond the activating enzyme breaks.
# Stereocentres are drawn with plain bonds (constitution only).
# ---------------------------------------------------------------------------
import math as _m
from matplotlib.patches import FancyArrowPatch as _Arrow, FancyBboxPatch as _Box

_PRO = COL["ester"]      # promoiety halo
_ACT = COL["oh"]         # group unmasked in the active drug


def _dir(a, d=1.0):
    return d * _m.cos(_m.radians(a)), d * _m.sin(_m.radians(a))


def _at(m, i, a, label="", d=1.0, order=1, hl=None, color=None, anchor=None):
    """New atom at angle a (deg) from atom i, bonded to it."""
    x, y = m.pos(i)
    dx, dy = _dir(a, d)
    k = m.add(x + dx, y + dy, label, color=color, anchor=anchor)
    m.bond(i, k, order, hl=hl)
    return k


def _phenyl_at(m, i, a):
    """Phenyl ring whose ipso carbon is one bond from atom i along angle a."""
    x, y = m.pos(i)
    cx, cy = x + 2 * _m.cos(_m.radians(a)), y + 2 * _m.sin(_m.radians(a))
    r = ring(m, cx, cy, 6, rot=a + 180)
    kekule(m, r, 0)
    m.bond(i, r[0])
    return r


def _cut(m, i, j, text="", dx=0.0, dy=-0.55, color=None):
    x1, y1 = m.pos(i); x2, y2 = m.pos(j)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    m.note(mx, my, "✂", color=color or COL["drug"], size=13)
    if text:
        m.note(mx + dx, my + dy, text, color=color or COL["drug"], size=7.5)


def _pair(top, bot, top_name, bot_name, enzyme, side_note, title, x_arrow=None, space=3.4, gap_names=0.0):
    """Stack prodrug (top) over active drug (bottom) at one scale, with a labelled arrow."""
    t0 = extent(top, pad=0.0); b0 = extent(bot, pad=0.0)
    dy = (t0[2] - space) - b0[3]
    m = Mol("pair")
    merge(m, top)
    merge(m, bot, 0, dy)
    xl = min(t0[0], b0[0])
    m.note(xl, t0[3] + 0.75, top_name, color=COL["ink"], size=9.5, ha="left", weight="bold")
    m.note(xl, b0[3] + dy + 0.75 + gap_names, bot_name, color=COL["ink"], size=9.5, ha="left", weight="bold")
    xa = x_arrow if x_arrow is not None else xl + 0.6
    y1, y2 = t0[2] - 0.25, b0[3] + dy + 1.35
    m.note(xa + 0.35, (y1 + y2) / 2 + 0.28, enzyme, color=COL["ester"], size=9, ha="left", weight="bold")
    m.note(xa + 0.35, (y1 + y2) / 2 - 0.28, side_note, color=COL["muted"], size=8.5, ha="left")

    def extra(ax):
        ax.add_patch(_Arrow((xa, y1), (xa, y2), arrowstyle="-|>", mutation_scale=15, lw=1.5, color=COL["ink"]))
    show(m, title, extra=extra)


# ---- enalapril / enalaprilat -----------------------------------------------
def enalapril(ester=True):
    m = Mol("enalapril" if ester else "enalaprilat")
    ph = ring(m, -C30, -0.5, 6, rot=30); kekule(m, ph, 0)
    p1 = m.add(C30, 0.5); m.bond(ph[0], p1)
    p2 = m.add(2 * C30, 0.0); m.bond(p1, p2)
    p3 = m.add(3 * C30, 0.5); m.bond(p2, p3)
    nh = m.add(4 * C30, 0.0, "NH"); m.bond(p3, nh)
    p5 = m.add(4 * C30, -1.0); m.bond(nh, p5)
    m.bond(p5, m.add(3 * C30, -1.5))                     # alanine CH3
    p6 = m.add(5 * C30, -1.5); m.bond(p5, p6)
    m.bond(p6, m.add(5 * C30, -2.5, "O"), 2)               # amide C=O
    R = 1 / (2 * _m.sin(_m.pi / 5))
    nx, ny = 6 * C30, -1.0
    pr = ring(m, nx + R * _m.cos(_m.radians(30)), ny + R * _m.sin(_m.radians(30)), 5, rot=210,
              labels=["N", "", "", "", ""])
    m.bond(p6, pr[0])
    ox, oy = outward(m, pr[1])
    a = _m.degrees(_m.atan2(oy, ox))
    c = _at(m, pr[1], a)
    _at(m, c, a + 60, "O", order=2)
    _at(m, c, a - 60, "OH")
    # the masked / unmasked carboxylate (zinc-binding group)
    ce = m.add(3 * C30, 1.5); m.bond(p3, ce)
    m.bond(ce, m.add(2 * C30, 2.0, "O"), 2)
    if ester:
        o = m.add(4 * C30, 2.0, "O"); m.bond(ce, o)
        c1 = m.add(5 * C30, 1.5); m.bond(o, c1, hl=_PRO)
        c2 = m.add(6 * C30, 2.0); m.bond(c1, c2, hl=_PRO)
        _cut(m, ce, o)
        m.note(5.5 * C30, 2.75, "ethyl promoiety", color=_PRO, size=8)
    else:
        o = m.add(4 * C30, 2.0, "OH", color=_ACT); m.bond(ce, o, hl=_ACT)
        m.note(5.9 * C30, 2.0, "Zn²⁺-binding carboxylate", color=_ACT, size=8, ha="left")
    return m


def fig_enalapril():
    _pair(enalapril(True), enalapril(False),
          "Enalapril (prodrug)  C₂₀H₂₈N₂O₅", "Enalaprilat (active)  C₁₈H₂₄N₂O₅",
          "carboxylesterase 1 (liver)", "+ H₂O  →  releases ethanol",
          "Enalapril → enalaprilat: an ethyl ester masks one carboxylic acid", x_arrow=-1.2)
    _md(["The ethyl ester (blue) removes one negative charge, so the prodrug is absorbed far better from the gut. "
         "Hepatic carboxylesterase cuts the acyl–oxygen bond (✂) and releases ethanol, a harmless leaving group. "
         "The freed carboxylate (red) is the group that binds the active-site **zinc ion of ACE**. "
         "*Three stereocentres (all S) are drawn with plain bonds.*"])


# ---- valacyclovir / acyclovir ---------------------------------------------
def _guanine(m):
    """Guanine, six-membered ring left, N9 lower right. Returns dict of key atoms."""
    r6 = ring(m, 0, 0, 6, rot=30, labels=["", "", "HN", "", "N", ""])  # C5, C6, N1, C2, N3, C4
    C5, C6, N1, C2, N3, C4 = r6
    r5 = fuse(m, C4, C5, 5, labels=["N", "", "N"])                      # from C5: N7, C8, N9
    N7, C8, N9 = r5[2], r5[3], r5[4]
    set_double(m, [(C2, N3), (C4, C5), (N7, C8)])
    m.bond(C6, m.add(m.pos(C6)[0], m.pos(C6)[1] + 1.0, "O"), 2)
    sub(m, C2, "H₂N", angle=210, anchor=2)
    return dict(N9=N9, C8=C8)


def valacyclovir(prodrug=True):
    m = Mol("valacyclovir" if prodrug else "acyclovir")
    g = _guanine(m)
    ids = chain(m, g["N9"], 5, heading=0, first=-1)     # CH2-O-CH2-CH2-O
    label(m, ids[2], "O")
    if not prodrug:
        label(m, ids[5], "OH")
        m.atoms[ids[5]]["color"] = _ACT
        m.bonds[-1]["hl"] = _ACT
        return m
    label(m, ids[5], "O")
    x, y = m.pos(ids[5])
    c = m.add(x + C30, y - 0.5); m.bond(ids[5], c)           # ester C=O carbon
    _at(m, c, 270, "O", order=2)
    ca = m.add(x + 2 * C30, y); m.bond(c, ca, hl=_PRO)       # valine C-alpha
    _at(m, ca, 90, "NH₂", hl=_PRO)
    cb = m.add(x + 3 * C30, y - 0.5); m.bond(ca, cb, hl=_PRO)
    _at(m, cb, -90, hl=_PRO)
    _at(m, cb, 30, hl=_PRO)
    _cut(m, ids[5], c)
    m.note(x + 3.3 * C30, y - 1.9, "L-valyl promoiety", color=_PRO, size=8, ha="left")
    return m


def fig_valacyclovir():
    _pair(valacyclovir(True), valacyclovir(False),
          "Valacyclovir (prodrug)  C₁₃H₂₀N₆O₄", "Acyclovir (released)  C₈H₁₁N₅O₃",
          "valacyclovir hydrolase (BPHL), gut wall and liver", "after uptake by the PEPT1 peptide transporter",
          "Valacyclovir → acyclovir: an amino-acid ester rides a nutrient transporter", x_arrow=-1.6)
    _md(["The L-valine ester (blue) makes the molecule look like a dipeptide, so the intestinal **PEPT1** transporter carries it in; "
         "oral bioavailability of acyclovir rises about three- to five-fold. An esterase-type hydrolase then cuts the ester (✂) and releases valine. "
         "Acyclovir is itself still a prodrug: **viral thymidine kinase** adds the first phosphate, host kinases add two more, and "
         "acyclovir triphosphate blocks the **viral DNA polymerase** (chain terminator). Because only infected cells make the first phosphate, "
         "uninfected cells are spared — activation, not binding, supplies the selectivity."])


# ---- levodopa / dopamine / carbidopa ---------------------------------------
def catechol_amine(kind):
    """kind: 'levodopa', 'dopamine', 'carbidopa'. Catechol ring left, side chain right."""
    m = Mol(kind)
    r = ring(m, 0, 0, 6, rot=0); kekule(m, r, 0)
    sub(m, r[2], "HO", angle=120, anchor=1)
    sub(m, r[3], "HO", angle=180, anchor=1)
    c1 = m.add(1 + C30, -0.5); m.bond(r[0], c1)
    c2 = m.add(1 + 2 * C30, 0.0); m.bond(c1, c2)
    if kind == "carbidopa":
        n = _at(m, c2, 90, "NH"); _at(m, n, 90, "NH₂")
        _at(m, c2, 270)                                       # alpha-methyl
    else:
        _at(m, c2, 90, "NH₂", hl=_ACT if kind == "dopamine" else None)
    if kind in ("levodopa", "carbidopa"):
        c3 = m.add(1 + 3 * C30, -0.5); m.bond(c2, c3, hl=_PRO if kind == "levodopa" else None)
        _at(m, c3, 330, "OH", hl=_PRO if kind == "levodopa" else None)
        _at(m, c3, 270, "O", order=2,
            hl=_PRO if kind == "levodopa" else None)
        if kind == "levodopa":
            _cut(m, c2, c3, "")
            m.note(1 + 3.6 * C30, 0.55, "lost as CO₂", color=_PRO, size=8, ha="left")
    return m


def fig_levodopa():
    _pair(catechol_amine("levodopa"), catechol_amine("dopamine"),
          "Levodopa (L-DOPA)  C₉H₁₁NO₄", "Dopamine (active)  C₈H₁₁NO₂",
          "aromatic L-amino-acid decarboxylase (AADC, PLP)", "inside the brain: −CO₂",
          "Levodopa → dopamine: the carboxyl group is the 'ticket' across the blood–brain barrier", x_arrow=-2.6)
    _md(["Dopamine cannot cross the blood–brain barrier. Levodopa is an amino acid, so the **LAT1** large neutral amino-acid "
         "transporter carries it into the brain (a *Trojan-horse* strategy). There **AADC** removes the carboxyl group as CO₂ (✂). "
         "No carrier group is cut off — the drug is made by a metabolic change — so levodopa behaves as a **bioprecursor**."])
    show(catechol_amine("carbidopa"), "Carbidopa  C₁₀H₁₄N₂O₄: a hydrazine AADC inhibitor that stays outside the brain")
    _md(["AADC is also present in gut wall, liver and blood vessels, so most of a levodopa dose would otherwise become dopamine in the periphery "
         "(nausea, low brain delivery). Carbidopa's hydrazine (–NH–NH₂) traps the PLP cofactor; it is too polar to cross the blood–brain barrier. "
         "The combination therefore protects the prodrug **until** it reaches the site where activation is wanted — "
         "a lesson that applies to any prodrug whose activating enzyme is widespread."])


# ---- oseltamivir / oseltamivir carboxylate ---------------------------------
def oseltamivir(ester=True):
    m = Mol("oseltamivir" if ester else "oseltamivir carboxylate")
    r = ring(m, 0, 0, 6, rot=0)          # C1 right, C2 upper right, C3 upper left, C4 left, C5 lower left, C6 lower right
    C1, C2, C3, C4, C5, C6 = r
    set_double(m, [(C1, C2)])
    c = _at(m, C1, 0)
    _at(m, c, 60, "O", order=2)
    if ester:
        o = _at(m, c, -60, "O")
        e1 = _at(m, o, 0, hl=_PRO); _at(m, e1, 60, hl=_PRO)
        _cut(m, c, o)
        m.note(m.pos(e1)[0] + 0.2, m.pos(e1)[1] - 0.6, "ethyl promoiety", color=_PRO, size=8)
    else:
        m.bond(c, m.add(m.pos(c)[0] + 0.5, m.pos(c)[1] - C30, "OH", color=_ACT), hl=_ACT)
    o3 = _at(m, C3, 120, "O")
    ch = _at(m, o3, 60)
    a1 = _at(m, ch, 120); _at(m, a1, 60)
    b1 = _at(m, ch, 0); _at(m, b1, 60)
    n = _at(m, C4, 180, "N")
    m.note(m.pos(n)[0], m.pos(n)[1] - 0.42, "H", color=COL["ink"], size=10.5)
    ac = _at(m, n, 150)
    _at(m, ac, 90, "O", order=2)
    _at(m, ac, 210)
    _at(m, C5, 240, "H₂N", anchor=2)
    return m


def fig_oseltamivir():
    _pair(oseltamivir(True), oseltamivir(False),
          "Oseltamivir (prodrug)  C₁₆H₂₈N₂O₄", "Oseltamivir carboxylate (active)  C₁₄H₂₄N₂O₄",
          "carboxylesterase 1 (liver)", "releases ethanol",
          "Oseltamivir → oseltamivir carboxylate: the same ethyl-ester trick on a neuraminidase inhibitor", x_arrow=-4.6)
    _md(["The carboxylate binds a cluster of arginines in the influenza **neuraminidase** active site, but as a free acid the drug "
         "is poorly absorbed. The ethyl ester raises oral bioavailability to roughly 80%; hepatic **CES1** removes it. "
         "Note the design is the same as enalapril: keep the pharmacophore, mask the one charge that blocks absorption. "
         "*Three ring stereocentres (3R,4R,5S) are drawn with plain bonds.*"])


# ---- fosphenytoin / phenytoin ---------------------------------------------
def hydantoin(stage):
    """stage: 'fos' (fosphenytoin), 'hm' (N-hydroxymethyl intermediate), 'phen' (phenytoin)."""
    m = Mol(stage)
    r = ring(m, 0, 0, 5, rot=180)            # v0 C5 (left), v1 N1, v2 C2, v3 N3, v4 C4
    C5, N1, C2, N3, C4 = r
    label(m, N1, "NH")
    _at(m, C4, 108, "O", order=2)
    _at(m, C2, 324, "O", order=2)
    _phenyl_at(m, C5, 158)
    _phenyl_at(m, C5, 236)
    if stage == "phen":
        label(m, N3, "NH")
        m.atoms[N3]["color"] = _ACT
        return m
    label(m, N3, "N")
    ch2 = _at(m, N3, 36, hl=_PRO)
    if stage == "hm":
        _at(m, ch2, -24, "OH", hl=_PRO)
        _cut(m, N3, ch2, "", color=COL["muted"])
        return m
    o = _at(m, ch2, -24, "O", hl=_PRO)
    p = m.add(m.pos(o)[0] + 1.35, m.pos(o)[1], "P", color=COL["phos"]); m.bond(o, p)
    m.bond(p, m.add(m.pos(p)[0], m.pos(p)[1] + 1.05, "O"), 2)
    m.bond(p, m.add(m.pos(p)[0], m.pos(p)[1] - 1.05, "OH"))
    m.bond(p, m.add(m.pos(p)[0] + 1.1, m.pos(p)[1], "OH"))
    _cut(m, o, p)
    return m


def fig_fosphenytoin():
    top, mid, bot = hydantoin("fos"), hydantoin("hm"), hydantoin("phen")
    scheme([top, ("alkaline phosphatase", "− phosphate"), mid, ("spontaneous", "− HCHO"), bot],
           title="Fosphenytoin → phenytoin: a tripartite prodrug with a self-immolative CH₂ linker", per_row=3, gap=3.4)
    _md(["**Left** fosphenytoin C₁₆H₁₅N₂O₆P · **middle** N-hydroxymethyl intermediate (not isolated) · **right** phenytoin C₁₅H₁₂N₂O₂.",
         "",
         "Phenytoin is too insoluble in water for a convenient intravenous solution. A phosphate makes the prodrug very water soluble, "
         "but a phosphate fixed directly on the imide nitrogen would be awkward to remove, so a one-carbon **spacer** (CH₂) sits between them. "
         "Phosphatases cut the P–O bond (✂); the N–CH₂OH that remains is unstable and falls apart on its own, releasing "
         "**formaldehyde** and phenytoin. That is the meaning of *self-immolative*: once the trigger is removed, the linker destroys itself."])


# ---- concept figure: four prodrug designs ----------------------------------
def fig_prodrug_types():
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10.5, 5.0))
    ax.set_xlim(0, 10.5); ax.set_ylim(0.6, 5.85); ax.axis("off")

    def box(x, y, w, txt, fc, ec, tc="white"):
        ax.add_patch(_Box((x, y), w, 0.5, boxstyle="round,pad=0.04,rounding_size=0.12", fc=fc, ec=ec, lw=1.2))
        ax.text(x + w / 2, y + 0.25, txt, ha="center", va="center", fontsize=9, color=tc, fontweight="bold")

    def cut(x, y):
        ax.text(x, y + 0.25, "✂", ha="center", va="center", fontsize=13, color=COL["drug"])

    rows = [
        (4.75, "A. Carrier-linked, bipartite",
         [("DRUG", 1.4, COL["drug"]), ("✂", 0, None), ("carrier (promoiety)", 2.2, COL["ester"])],
         "carrier joined directly to the drug (ester, amide, phosphate); one enzyme step frees the drug  —  enalapril, valacyclovir"),
        (3.45, "B. Carrier-linked, tripartite",
         [("DRUG", 1.4, COL["drug"]), ("linker", 1.0, COL["muted"]), ("✂", 0, None), ("carrier", 1.4, COL["ester"])],
         "enzyme cuts the carrier; the exposed linker then falls off by itself (self-immolative)  —  fosphenytoin, pivoxil esters"),
        (2.15, "C. Mutual prodrug",
         [("DRUG A", 1.4, COL["drug"]), ("✂", 0, None), ("DRUG B", 1.4, COL["phos"])],
         "two active drugs linked so each is the other's carrier  —  sultamicillin (ampicillin + sulbactam)"),
        (0.85, "D. Bioprecursor",
         [("inactive precursor", 2.2, COL["muted"]), ("→", 0, None), ("DRUG", 1.4, COL["drug"])],
         "no carrier to remove: oxidation, reduction, decarboxylation or phosphorylation creates the drug  —  levodopa, clopidogrel"),
    ]
    for y, head, parts, note in rows:
        ax.text(0.1, y + 0.78, head, fontsize=10.5, fontweight="bold", color=COL["ink"], va="center")
        x = 0.3
        for txt, w, col in parts:
            if txt == "✂":
                cut(x + 0.22, y); x += 0.45
            elif txt == "→":
                ax.annotate("", xy=(x + 0.75, y + 0.25), xytext=(x + 0.05, y + 0.25),
                            arrowprops=dict(arrowstyle="-|>", color=COL["ink"], lw=1.4))
                ax.text(x + 0.4, y + 0.55, "enzyme", ha="center", fontsize=7.5, color=COL["ester"])
                x += 0.85
            else:
                box(x, y, w, txt, col, col)
                if txt == "DRUG" and parts[1][0] == "linker":
                    ax.plot([x + w + 0.02, x + w + 0.16], [y + 0.25, y + 0.25], color=COL["ink"], lw=2)
                    x += w + 0.18
                else:
                    x += w + 0.05
        import textwrap as _tw
        ax.text(5.6, y + 0.25, _tw.fill(note, 58), fontsize=8.6, color=COL["muted"], va="center", ha="left",
                linespacing=1.2)
    ax.set_title("Four ways to build a prodrug", loc="left", fontsize=12, fontweight="bold")
    plt.show()


# ---- calculators -----------------------------------------------------------
def activation_calc(t_half_typical_h=1.5, time_h=2.0, activity_low=0.3, activity_high=3.0):
    """First-order activation (prodrug -> drug) for a typical patient and two outliers."""
    import matplotlib.pyplot as plt
    import numpy as np
    k0 = _m.log(2) / t_half_typical_h
    rows, out = [], {}
    lines = ["**Model.** Activation is first order: [prodrug]ₜ = [prodrug]₀·e^(−kt), so % converted = 100·(1 − e^(−kt)).",
             "",
             "1. Typical patient: k = ln 2 / t½ = 0.693 / %.2f h = **%.3f h⁻¹**" % (t_half_typical_h, k0)]
    step = 2
    for name, f in (("low-esterase patient", activity_low), ("typical patient", 1.0), ("high-esterase patient", activity_high)):
        k = k0 * f
        th = _m.log(2) / k if k > 0 else float("inf")
        pc = 100 * (1 - _m.exp(-k * time_h))
        out[name] = dict(k=k, t_half=th, pct=pc)
        if f != 1.0:
            lines.append("%d. %s (enzyme activity × %.2g): k = %.3f × %.2g = **%.3f h⁻¹**, t½ = 0.693 / k = **%.2f h**"
                         % (step, name.capitalize(), f, k0, f, k, th))
            step += 1
        rows.append("| %s | %.2g | %.3f | %.2f | %.1f |" % (name, f, k, th, pc))
    lines.append("%d. Fraction converted at t = %.1f h: 100·(1 − e^(−k·%.1f)) for each patient (table)." % (step, time_h, time_h))
    spread = out["high-esterase patient"]["pct"] - out["low-esterase patient"]["pct"]
    lines += ["", "| Patient | activity × | k (h⁻¹) | t½ (h) | %% active drug at %.1f h |" % time_h, "|---|---|---|---|---|"] + rows
    lines += ["", "**Between-patient spread at %.1f h = %.1f − %.1f = %.1f percentage points.**"
              % (time_h, out["high-esterase patient"]["pct"], out["low-esterase patient"]["pct"], spread)]
    _md(lines)
    t = np.linspace(0, max(4 * t_half_typical_h / max(activity_low, 1e-3) / 2, 2 * time_h), 300)
    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    for (name, d), col in zip(out.items(), (COL["dbl"], COL["ester"], COL["phos"])):
        ax.plot(t, 100 * (1 - np.exp(-d["k"] * t)), lw=2.2, color=col, label="%s (t½ %.2f h)" % (name, d["t_half"]))
    ax.axvline(time_h, color=COL["muted"], ls="--", lw=1)
    ax.set_xlabel("time after the dose (h)"); ax.set_ylabel("% converted to active drug")
    ax.set_ylim(-2, 102); ax.grid(alpha=.25); ax.legend(fontsize=8, loc="lower right")
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    ax.set_title("First-order activation in three patients", fontsize=10.5, loc="left")
    plt.tight_layout(); plt.show()
    out["spread_pct_points"] = spread
    return out


def logD_ester_calc(logP_acid=1.0, pKa_acid=4.0, logP_ester=2.3, pH=6.5):
    """Acid vs its ethyl ester: log D at a chosen pH (ester has no ionizable group)."""
    ratio = 10 ** (pH - pKa_acid)
    corr = _m.log10(1 + ratio)
    logD_acid = logP_acid - corr
    logD_ester = logP_ester
    diff = logD_ester - logD_acid
    _md(["*Illustrative values for a generic carboxylic-acid drug and its ethyl ester (not measured data for a named drug).*",
         "",
         "1. Acid: [A⁻]/[HA] = 10^(pH − pKa) = 10^(%.1f − %.1f) = **%.3g** → %.2f%% ionized" % (pH, pKa_acid, ratio, 100 * ratio / (1 + ratio)),
         "2. Acid: log D = log P − log₁₀(1 + 10^(pH − pKa)) = %.2f − %.2f = **%.2f**" % (logP_acid, corr, logD_acid),
         "3. Ethyl ester: no ionizable group, so log D = log P = **%.2f** at any pH" % logD_ester,
         "4. Difference: %.2f − (%.2f) = **%.2f log units** → the ester partitions about **%.0f-fold** more into octanol at pH %.1f"
         % (logD_ester, logD_acid, diff, 10 ** diff, pH),
         "",
         "Most of the gain comes from removing the charge (step 2), not from the two extra carbons (log P %.2f → %.2f)." % (logP_acid, logP_ester)])
    return dict(ratio=ratio, logD_acid=logD_acid, logD_ester=logD_ester, difference=diff, fold=10 ** diff)

import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110})
# ===== END kit =====


## Learning objectives

| # | By the end of this activity you should be able to | Bloom | Syllabus level | SILO |
|---|---|---|---|---|
| 1 | Predict how varying the promoiety changes the rate of hydrolytic activation. | Apply | IV | 4 |
| 2 | Distinguish carrier-linked prodrugs, bipartite designs, mutual prodrugs and bioprecursors. | Understand | V | 5 |
| 3 | Evaluate a proposed prodrug for activation reliability and patient variability. | Evaluate | II | 4 |
| 4 | Design a prodrug for a stated problem and name the activating enzyme. | Create | I | 5 |

*Syllabus levels follow the PHS 312 legend, in which Level I is Creating and Level VI is Remembering.*

## Part 1 - What do you already know?

Three questions before you begin. **You will not be told whether you are
right**, and that is deliberate: the point is to measure what you knew coming
in, so that at the end you can see how much the activity moved you. Answer
from what you remember from the reading. Guessing is fine.

In [ ]:
#@title P1   (run after choosing) { display-mode: "form" }
#@markdown **P1.**  In a carrier-linked prodrug, what does the promoiety do?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It protects the drug from the immune system during transport", "B. It binds the target directly once the drug has been absorbed", "C. It changes the physical properties and is later cleaved off", "D. It provides the pharmacological activity until the drug acts"]
gb.answer("P1", response[:1])

In [ ]:
#@title P2   (run after choosing) { display-mode: "form" }
#@markdown **P2.**  An ester prodrug is activated faster when the alcohol carries an electron-withdrawing aryl group. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The compound becomes more water soluble and so hydrolyses", "B. The ester bond becomes longer and therefore weaker overall", "C. The aryl group attracts esterase enzymes to the molecule", "D. The carbonyl becomes more electrophilic and easier to attack"]
gb.answer("P2", response[:1])

In [ ]:
#@title P3   (run after choosing) { display-mode: "form" }
#@markdown **P3.**  A prodrug behaves consistently in plasma but gives variable exposure across patients. What does that suggest?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Activation depends on an enzyme whose activity differs between people", "B. The prodrug is chemically unstable and decomposes before it is taken", "C. The assay used to measure plasma concentrations is poorly calibrated", "D. The drug itself has a narrow therapeutic window in these patients"]
gb.answer("P3", response[:1])

## Part 2 - Key ideas: what a prodrug is and why you would make one


**Summary.** A **prodrug** is a compound with little or no activity of its own that the body converts into the active drug. The pharmacophore is kept intact; a removable group (the **promoiety** or **carrier**) is attached to fix a problem that has nothing to do with binding — the drug is too polar to be absorbed, too insoluble to inject, too short-lived, too bitter, too unstable, or active in the wrong tissue. An enzyme (sometimes just pH or light) then removes or changes the masking group where and when the drug is needed.

The design only works if three things are true: (1) the prodrug survives long enough to do its job (cross the gut wall, dissolve, reach the tissue); (2) activation is **fast and reliable** once it gets there; (3) whatever falls off is **harmless**.

| Problem with the drug | What the prodrug changes | Example (activation) |
|---|---|---|
| Poor oral absorption (charged acid, phosphate or very polar group) | masks the charge with an ester; raises log D | enalapril, oseltamivir (liver carboxylesterase); tenofovir disoproxil |
| Cannot reach the target tissue (e.g. blood–brain barrier) | borrows a nutrient transporter (*Trojan horse*) | levodopa via LAT1; valacyclovir via PEPT1 |
| Too short a duration of action | slow-release reservoir or slowly cleaved group | bambuterol → terbutaline (slow plasma cholinesterase hydrolysis) |
| Side effects or toxicity at the wrong site | drug is released mainly where an activating enzyme is concentrated | capecitabine → 5-FU (tumour thymidine phosphorylase); acyclovir (viral kinase) |
| Bad taste in an oral liquid | lowers water solubility so less dissolves on the tongue | erythromycin esters in paediatric suspensions |
| Too insoluble for an injection | adds an ionizable phosphate or amine | fosphenytoin; irinotecan |
| Chemically unstable in the bottle | locks a reactive group into a ring that reopens in the body | the reading's §14.6.7 example |
| Needs switching on at one site | external trigger (light) — the reading's "sleeping agents" | 5-aminolevulinic acid → protoporphyrin IX, a photosensitizer for photodynamic therapy |

In [ ]:
#@title 2.1 Figure — four ways to build a prodrug { display-mode: "form" }
fig_prodrug_types()

| Class | What is attached | How the drug is released | What to look for in a structure |
|---|---|---|---|
| **Carrier-linked, bipartite** | carrier bonded straight to the drug | one hydrolysis (esterase, phosphatase, peptidase) | an ester, amide, carbonate or phosphate on a group the drug needs for binding |
| **Carrier-linked, tripartite** | carrier → **linker (spacer)** → drug | enzyme removes the carrier; the linker then breaks off **spontaneously** (self-immolative) | –O–CH₂–O– or –N–CH₂–O– acetal-type links; often release formaldehyde or acetaldehyde |
| **Mutual prodrug** | a second drug acts as the carrier | one cleavage releases two active compounds | two recognizable drugs joined by an ester or azo bond |
| **Bioprecursor** | nothing — the molecule itself is changed | oxidation, reduction, decarboxylation, phosphorylation or an acid-catalysed rearrangement | the active species is a *metabolite*; no carrier is shed |

Why bother with a linker? Some bonds are hard for enzymes to reach (a crowded or weakly nucleophilic atom, such as the imide N of phenytoin) or would release a toxic carrier. A spacer moves the enzyme's cut away from the drug, and the left-over fragment is designed to be unstable, so it falls apart as soon as the carrier is gone.

<details>
<summary><b>▸ Key terms (open)</b></summary>


| Term | Meaning |
|---|---|
| Prodrug | Inactive (or much less active) compound converted in the body to the active drug |
| Promoiety / carrier | The removable group attached to the drug |
| Bipartite prodrug | Drug and carrier joined directly |
| Tripartite prodrug | Drug – linker (spacer) – carrier |
| Self-immolative linker | A spacer that decomposes spontaneously once the carrier has been cleaved |
| Mutual prodrug | Two pharmacologically active agents joined so that each carries the other |
| Bioprecursor | A prodrug with no carrier; it becomes active by a metabolic change to its own structure |
| Trojan-horse strategy | Making the drug resemble a nutrient so a membrane transporter carries it in |
| Activation | The chemical or enzymatic step that releases or creates the active drug |
| Carboxylesterase (CES1, CES2) | Serine hydrolases that cut esters, amides and carbamates; CES1 mainly liver, CES2 mainly intestine |
| Sleeping agent | The reading's term for a prodrug switched on by an external influence such as light |
| Soft drug | The reverse idea: an *active* drug designed to be inactivated quickly by metabolism |

</details>

#### Worked example: classifying four prodrugs
Classify each as bipartite, tripartite, mutual or bioprecursor, and name what is released besides the drug: (a) oseltamivir; (b) tenofovir disoproxil (each phosphonate OH is masked as –O–CH₂–O–C(=O)–O–iPr); (c) sultamicillin (ampicillin and sulbactam joined through a –O–CH₂–O– diester); (d) clopidogrel.

<details>
<summary><b>▸ Worked solution</b></summary>

1. **Oseltamivir — bipartite.** An ethyl ester sits directly on the carboxylate; CES1 releases **ethanol**.
2. **Tenofovir disoproxil — tripartite.** Esterases cut the carbonate first; the exposed –O–CH₂–O–P hemiacetal then collapses on its own. Each arm releases **isopropanol, CO₂ and formaldehyde**.
3. **Sultamicillin — mutual prodrug** (with a methylene linker). Hydrolysis releases **ampicillin** (antibiotic) and **sulbactam** (β-lactamase inhibitor) plus formaldehyde; each drug is the other's carrier.
4. **Clopidogrel — bioprecursor.** Nothing is cut off as a carrier; CYP450 oxidation (mainly CYP2C19) converts the thiophene ring, in two steps, into the active thiol.

Rule of thumb: if you can draw the drug and a separate fragment that leaves, it is carrier-linked; if the active species is a *metabolite* of the whole molecule, it is a bioprecursor.
</details>

## Part 3 - Masking a charge: ester and amino-acid prodrugs on one template


The commonest prodrug problem is a **carboxylic acid** (or phosphate) that the target needs for an ionic or metal contact but that is almost fully ionized in the small intestine, so the drug crosses membranes poorly. Converting the acid to an **ester** removes the charge; esterases put it back after absorption. Three design rules follow from the chemistry of ester hydrolysis:

* **Electron-withdrawing groups on the alcohol speed hydrolysis** (they make the carbonyl carbon more electrophilic and the alkoxide a better leaving group): aryl esters such as 5-indanyl are cleaved faster than ethyl esters.
* **Steric bulk slows hydrolysis**: a pivaloyl or tert-butyl group shields the carbonyl from the esterase serine.
* **The leaving alcohol must be safe** at the amount released: ethanol, valine and 5-indanol are; methanol is avoided.

Below, each prodrug sits directly above its active drug on the same template. Only the blue promoiety changes; the scissors mark the bond the enzyme breaks.

In [ ]:
#@title 3.1 Figure — enalapril above enalaprilat { display-mode: "form" }
fig_enalapril()

In [ ]:
#@title 3.2 Figure — oseltamivir above oseltamivir carboxylate { display-mode: "form" }
fig_oseltamivir()

In [ ]:
#@title 3.3 Figure — valacyclovir above acyclovir { display-mode: "form" }
fig_valacyclovir()

#### Worked example: how much does an ester raise log D?
A carboxylic-acid drug has log P = 1.0 and pKa = 4.0. Its ethyl ester has log P = 2.3 and no ionizable group (illustrative values). Calculate log D for each at pH 6.5 (upper small intestine) and the fold-change in octanol/water distribution.

<details>
<summary><b>▸ Worked solution</b></summary>

log D = log P − log₁₀(1 + 10^(pH − pKa)) for an acid.

1. Acid: 10^(6.5 − 4.0) = 10^2.5 = 316, so the acid is 316/317 = **99.7% ionized**.
2. Acid: log D = 1.0 − log₁₀(317) = 1.0 − 2.50 = **−1.50**.
3. Ester: nothing ionizes, so **log D = log P = 2.30**.
4. Difference = 2.30 − (−1.50) = **3.8 log units ≈ 6300-fold** more partitioning into the lipid phase.

Only 1.3 of the 3.8 units come from the extra carbons; the other 2.5 come from removing the charge. Check with the calculator below.
</details>

In [ ]:
#@title 3.4 Calculator — log D of an acid and its ethyl ester { display-mode: "form" }
logP_acid = 1.0 #@param {type:"number"}
pKa_acid = 4.0 #@param {type:"number"}
logP_ester = 2.3 #@param {type:"number"}
pH = 6.5 #@param {type:"slider", min:1.0, max:8.0, step:0.1}
_ = logD_ester_calc(logP_acid, pKa_acid, logP_ester, pH)

**Try it.** Set pH to 1.5 (stomach): the acid is now mostly un-ionized and the ester's advantage shrinks to its log P difference. The prodrug matters most where the acid is charged — the intestine and blood.

## Part 4 - Bioprecursors, self-immolative linkers and how fast activation happens


Not every prodrug sheds a carrier. A **bioprecursor** becomes active because metabolism rewrites part of the molecule. Levodopa is the classic case: removing CO₂ creates dopamine, and the carboxyl group that is lost is exactly what lets an amino-acid transporter carry the molecule into the brain.

In [ ]:
#@title 4.1 Figure — levodopa above dopamine, and carbidopa { display-mode: "form" }
fig_levodopa()

A **tripartite** prodrug puts a spacer between drug and carrier. Fosphenytoin is the cleanest example: the phosphate (carrier) gives water solubility for an intravenous solution, the CH₂ is the self-immolative linker, and phenytoin is the drug.

In [ ]:
#@title 4.2 Figure — fosphenytoin to phenytoin in two steps { display-mode: "form" }
fig_fosphenytoin()

**Where activation happens decides what the prodrug can do.**

| Activating agent | Main location | What it does | Examples |
|---|---|---|---|
| Carboxylesterase 1 (CES1) | liver | hydrolyses esters with a large acyl part and small alcohol | enalapril, oseltamivir, capecitabine (first step) |
| Carboxylesterase 2 (CES2) | small intestine (also liver) | prefers a small acyl part and large alcohol | irinotecan → SN-38 |
| Plasma cholinesterase and other blood esterases | blood | slow hydrolysis of carbamates and some esters | bambuterol → terbutaline |
| Alkaline phosphatase | intestinal brush border, liver, plasma | removes phosphate monoesters | fosphenytoin, fosamprenavir |
| Amino-acid ester hydrolase (valacyclovirase) | intestine, liver | cuts amino-acid esters | valacyclovir, valganciclovir |
| Cytochrome P450 | liver, gut wall | oxidation, O- and N-dealkylation | clopidogrel (CYP2C19), codeine → morphine (CYP2D6) |
| Bacterial azoreductase | colon | reduces N=N | balsalazide, olsalazine → 5-aminosalicylic acid |
| Aromatic L-amino-acid decarboxylase | brain and periphery | removes CO₂ | levodopa → dopamine |
| Kinases (viral thymidine kinase first) | infected cells | add phosphate | acyclovir → acyclovir triphosphate |
| Acid (no enzyme) | acidic compartments | rearrangement | omeprazole → sulfenamide in the parietal-cell canaliculus |

**What can go wrong.**
* **Variable activation.** If the activating enzyme is polymorphic or easily inhibited, exposure varies between patients: CYP2C19 poor metabolizers make less active clopidogrel metabolite; CYP2D6 ultrarapid metabolizers make morphine from codeine faster than expected; some CES1 variants hydrolyse ester prodrugs more slowly.
* **Species differences.** Rodent plasma is rich in carboxylesterase and human plasma is not, so an ester that is cleaved instantly in rat blood may survive much longer in a patient.
* **Activation in the wrong place.** Peripheral AADC wastes levodopa (hence carbidopa); an ester hydrolysed in the gut lumen loses its absorption advantage.
* **Toxic or burdensome by-products.** Pivoxil (pivaloyloxymethyl) esters release pivalic acid, which is excreted bound to carnitine and can deplete carnitine during long courses; acyloxymethyl and N-CH₂ linkers release small amounts of formaldehyde. The promoiety must be judged at the amount actually released.
* **Incomplete activation.** Some prodrug may be excreted unchanged, so the dose of prodrug is not the dose of drug.

#### Worked example: how reliably does an ester switch on?
An ester prodrug is activated by first-order hydrolysis with t½ = 1.5 h in a typical patient. Patient L has 0.3 × typical esterase activity and patient H has 3 ×. What percentage of the dose is active drug in each at 2 h, and what is the between-patient spread? Repeat for a much faster promoiety with t½ = 0.25 h.

<details>
<summary><b>▸ Worked solution</b></summary>

1. Typical k = ln 2 / t½ = 0.693 / 1.5 = **0.462 h⁻¹**.
2. Patient L: k = 0.3 × 0.462 = 0.139 h⁻¹; % at 2 h = 100(1 − e^(−0.277)) = **24.2%**.
3. Patient H: k = 3 × 0.462 = 1.386 h⁻¹; % at 2 h = 100(1 − e^(−2.773)) = **93.8%**.
4. Spread = 93.8 − 24.2 = **69.5 percentage points** (typical patient 60.3%).
5. Faster promoiety, t½ = 0.25 h: typical k = 2.77 h⁻¹; L: k = 0.832 h⁻¹ → **81.1%**; H: k = 8.32 h⁻¹ → **≈100.0%**; spread ≈ **19 points**.

The same tenfold range of enzyme activity produces a much smaller difference in drug delivered when activation is fast relative to the time point of interest. Reproduce both cases with the calculator (set t½ to 0.25).
</details>

In [ ]:
#@title 4.3 Calculator — first-order activation in two outlier patients { display-mode: "form" }
t_half_typical_h = 1.5 #@param {type:"slider", min:0.1, max:12, step:0.05}
time_h = 2.0 #@param {type:"slider", min:0.5, max:12, step:0.5}
activity_low = 0.3 #@param {type:"number"}
activity_high = 3.0 #@param {type:"number"}
_ = activation_calc(t_half_typical_h, time_h, activity_low, activity_high)

## Part 5 - How fast does the promoiety come off?

A carrier-linked ester prodrug is activated by hydrolysis. The left panel
shows prodrug disappearing and active drug appearing. The right panel shows
the **same prodrug in three patients** whose esterase activity differs
threefold either way.

Work through the promoieties from fastest to slowest and watch both panels.

<details>
<summary><b>Stuck? Open this before you ask a neighbour</b></summary>

The two panels answer different questions, and the second is the one that
matters clinically.

* **Left panel: how fast?** An electron-withdrawing 5-indanyl ester is
  essentially fully activated within two hours. A hindered pivaloyl ester
  takes most of a working day. An amide barely moves.
* **Right panel: how *reliably* fast?** Look at the spread between the three
  patients at two hours. A very fast promoiety is nearly complete in all
  three, so the spread is small. A very slow one has barely started in any of
  them, so the spread is also small. It is the *intermediate* rates where the
  three patients diverge most — and that is where dosing becomes
  unpredictable.

Read `patient spread at 2 h` for each promoiety before you decide which you
would choose.

</details>

In [ ]:
#@title Activate the prodrug { display-mode: "form" }
#@markdown Choose a promoiety and the patient's enzyme activity.
promoiety = "ethyl ester" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide", "no promoiety"]
patient_esterase_activity = 1.0 #@param {type:"slider", min:0.1, max:3, step:0.1}
hours_followed = 8 #@param {type:"slider", min:1, max:24, step:1}
r = sims.prodrug_kinetics(promoiety, patient_esterase_activity, hours_followed)
print(f"rate constant k        {r['k_per_h']:.3f} per hour")
hl = r['half_life_h']
print(f"half-life of activation{'  ' + format(hl, '.2f') + ' h' if hl else '  effectively none'}")
print(f"active drug at 2 h     {r['pct_active_at_2h']:.1f}% of dose")
print(f"patient spread at 2 h  {r['patient_spread_at_2h_pct']:.1f} percentage points")

**Now put two settings side by side.** Run the cell below and compare the two figures it prints, one above the other.

In [ ]:
#@title Side by side: a fast, a moderate and a slow promoiety { display-mode: "form" }
#@markdown Three promoieties, same drug, same patients.
fast_option = "5-indanyl ester (EWG)" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
middle_option = "ethyl ester" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
slow_option = "pivaloyl ester (hindered)" #@param ["ethyl ester", "5-indanyl ester (EWG)", "pivaloyl ester (hindered)", "amide"]
out = []
for p in (fast_option, middle_option, slow_option):
    print("=" * 70); print(p.upper()); print("=" * 70)
    out.append((p, sims.prodrug_kinetics(p, 1.0, 8.0)))
print()
print(f"{'promoiety':<30}{'k /h':>8}{'t1/2 h':>9}{'%@2h':>8}{'spread':>9}")
for p, r in out:
    hl = f"{r['half_life_h']:.2f}" if r['half_life_h'] else "-"
    print(f"{p:<30}{r['k_per_h']:>8.2f}{hl:>9}{r['pct_active_at_2h']:>8.1f}"
          f"{r['patient_spread_at_2h_pct']:>8.1f}%")
print("\nThe largest between-patient spread is not at the fastest or the slowest rate.")

In [ ]:
#@title C1   (run after choosing) { display-mode: "form" }
#@markdown **C1.**  The ethyl ester showed a far larger between-patient spread at two hours than the indanyl ester did. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. At an intermediate rate, two hours falls where the curves differ most", "B. The ethyl ester is chemically unstable and decomposes unpredictably", "C. Esterase activity varies more widely for ethyl esters than aryl ones", "D. The indanyl ester is activated by a route that needs no enzyme at all"]
gb.answer("C1", response[:1])

In [ ]:
#@title C2   (run after choosing) { display-mode: "form" }
#@markdown **C2.**  You must choose a promoiety for a drug with a narrow therapeutic window. Which consideration should dominate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Reliability of activation across patients, not raw speed alone", "B. The lowest possible rate constant, to avoid any sudden release", "C. The highest possible rate constant, to minimise exposure time", "D. The lowest molecular weight promoiety that can be attached"]
gb.answer("C2", response[:1])

## Part 6 - Prodrugs in practice (reference)


| Prodrug | Active drug | Promoiety / activation | Reason for the prodrug | Target / use |
|---|---|---|---|---|
| **Enalapril** | enalaprilat | ethyl ester; hepatic CES1 | oral absorption | angiotensin-converting enzyme (zinc metalloprotease); hypertension, heart failure |
| **Oseltamivir** | oseltamivir carboxylate | ethyl ester; hepatic CES1 | oral absorption | influenza neuraminidase; influenza A and B |
| **Valacyclovir** | acyclovir → acyclovir triphosphate | L-valyl ester; PEPT1 uptake, valacyclovirase; then viral thymidine kinase | oral absorption; activation only in infected cells | herpesvirus DNA polymerase; HSV and VZV infections |
| **Levodopa** (+ carbidopa) | dopamine | none (bioprecursor); LAT1 transport, AADC decarboxylation | crossing the blood–brain barrier | dopamine receptors; Parkinson's disease |
| **Fosphenytoin** | phenytoin | phosphate + CH₂ linker (tripartite); phosphatases, then spontaneous loss of formaldehyde | water solubility for IV use | voltage-gated Na⁺ channels; status epilepticus |
| **Tenofovir disoproxil** | tenofovir (→ diphosphate) | two carbonate–acetal arms (tripartite); esterases | oral absorption of a phosphonate | HIV reverse transcriptase, HBV polymerase |
| **Candesartan cilexetil** | candesartan | carbonate–acetal ester (tripartite); hydrolysed in the gut wall during absorption | oral absorption | AT₁ angiotensin II receptor; hypertension |
| **Capecitabine** | 5-fluorouracil | carbamate + sugar; CES1 → cytidine deaminase → thymidine phosphorylase (high in many tumours) | oral use; more drug released in tumour | thymidylate synthase (as FdUMP); colorectal and breast cancer |
| **Irinotecan** | SN-38 | piperidinopiperidine carbamate; CES2 | water solubility for IV use | topoisomerase I; colorectal cancer |
| **Bambuterol** | terbutaline | bis-dimethylcarbamate; plasma cholinesterase (slow) | long duration | β₂-adrenergic receptor agonist; asthma |
| **Clopidogrel** | active thiol metabolite | none (bioprecursor); CYP2C19 and other CYPs, two oxidations | — (discovered as such) | P2Y₁₂ ADP receptor, irreversible; prevention of arterial thrombosis |
| **Omeprazole** | sulfenamide | none; acid-catalysed rearrangement | activation only in the acidic parietal-cell canaliculus | H⁺/K⁺-ATPase (covalent); peptic ulcer, reflux |

**Looking back (L31 Bioisosteres).** A bioisostere changes a group **permanently** to fix a property; a prodrug changes it **temporarily** and relies on the body to undo the change. The two are often combined: candesartan keeps the **tetrazole** bioisostere for its acid contact and adds a **cilexetil ester** prodrug on its carboxylic acid for oral absorption. When a property problem appears in a lead, ask both questions — can the group be replaced, or only hidden until absorption?

## Part 7 - End-of-activity assessment

Five questions in the style of the course quizzes and exams, mapped to the
Student Intended Learning Outcomes for this unit (**SILO 4**).
This time you get full feedback, including why each wrong option is wrong.
Answer each one, then run the summary cell at the bottom.

In [ ]:
#@title Q1   (run after choosing) { display-mode: "form" }
#@markdown **Q1.**  SAR data: OCH2 active, NHCH2 active, SCH2 inactive, CH2CH2 inactive, CH=CH inactive. What does the pattern establish about the oxygen?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. It contributes bulk that the binding site sterically requires", "B. It stabilizes the active conformation through its bond angle", "C. It raises lipophilicity enough to drive membrane permeation", "D. It acts as a hydrogen-bond acceptor at that position in the site"]
gb.answer("Q1", response[:1])

In [ ]:
#@title Q2   (run after choosing) { display-mode: "form" }
#@markdown **Q2.**  A CF3 group is introduced two bonds from a basic amine that must engage an anionic site residue. Potency falls. Why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. CF3 sterically blocks the amine's approach to the site residue", "B. CF3 lowers the amine's pKa, reducing the protonated fraction", "C. CF3 is metabolized to an acid that competes for the same site", "D. CF3 raises lipophilicity, which excludes the drug from plasma"]
gb.answer("Q2", response[:1])

In [ ]:
#@title Q3   (run after choosing) { display-mode: "form" }
#@markdown **Q3.**  Two esters of one drug differ only in the alcohol: ethanol versus 5-indanol. How will activation compare, and why?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The indanyl ester hydrolyzes faster, as the aryl group withdraws density", "B. Neither hydrolyzes in plasma, since aryl esters require liver enzymes", "C. Both hydrolyze identically, since esterases are insensitive to structure", "D. The ethyl ester hydrolyzes faster, as smaller esters are less hindered"]
gb.answer("Q3", response[:1])

In [ ]:
#@title Q4   (run after choosing) { display-mode: "form" }
#@markdown **Q4.**  Chloramphenicol is formulated as the palmitate ester for oral suspension and the succinate ester for injection. What does this contrast demonstrate?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. Two promoieties are required before any prodrug can be marketed", "B. Ester prodrugs invariably increase the aqueous solubility of a drug", "C. Promoiety choice is dictated by the route and the dosage form needed", "D. The parent drug is inactive until both esters have been hydrolyzed"]
gb.answer("Q4", response[:1])

In [ ]:
#@title Q5   (run after choosing) { display-mode: "form" }
#@markdown **Q5.**  A prodrug performs consistently in plasma incubations but gives highly variable exposure across patients. Which explanation fits best?
#@markdown ---
response = "-- choose --" #@param ["-- choose --", "A. The promoiety itself is pharmacologically active in some patients", "B. Activating enzyme or gut flora activity differs between individuals", "C. The linkage hydrolyzes chemically rather than enzymatically in vivo", "D. Plasma incubation overestimates the drug's intrinsic potency in vivo"]
gb.answer("Q5", response[:1])

## Your results

In [ ]:
#@title Show my performance and my report code { display-mode: "form" }
gb.summary()
print()
_ = gb.report_code()

### What to submit

1. Copy the single `L32|...` line printed above.
2. Paste it into the Canvas assignment **L32 Colab activity**.
3. With your partner, be ready to say one thing the simulation showed you that
   the reading did not.

Your report code records how many questions you answered correctly. It is a
participation and self-check record, not a proctored exam score.